# XGBoost classification

**Train initial XGBoost models for each configuration**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef
)

# Define project directories
PROJECT_DIR = Path(r"C:\UTAR - Master\Master Project\Project\Main Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

# Load chronological model development data
data = pd.read_csv(
    PROCESSED_DIR / "triple_barrier_model_development.csv",
    parse_dates=["Date", "Event_End_Date"]
)

# Define candidate model features
feature_columns = [
    "returns",
    "returns_lag_1",
    "returns_lag_2",
    "momentum_5",
    "momentum_10",
    "daily_range",
    "candle_body",
    "sma_5_ratio",
    "sma_10_ratio",
    "sma_20_ratio",
    "volatility_5",
    "volatility_10",
    "volatility_20",
    "rsi",
    "macd",
    "macd_signal",
    "macd_diff",
    "volume_change",
    "volume_ratio_20",
    "bb_high",
    "bb_low",
    "bb_width",
    "skewness_20"
]

# Verify dataset structure and chronological separation
assert len(feature_columns) == 23
assert data[feature_columns].notna().all().all()
assert data["Configuration"].nunique() == 4
assert data["Label"].isin([-1, 0, 1]).all()

assert (
    data.loc[
        data["Split"] == "Train_2015_2022",
        "Event_End_Date"
    ] <= pd.Timestamp("2022-12-31")
).all()

assert (
    data.loc[
        data["Split"] == "Validation_2023_2024",
        "Date"
    ] >= pd.Timestamp("2023-01-01")
).all()

assert (data["Date"] < pd.Timestamp("2025-01-01")).all()

# Define consistent model parameters
model_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 3,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

results = []
prediction_records = []
trained_models = {}

# Train separate models for each market and configuration
for configuration in sorted(data["Configuration"].unique()):

    for country in ["Malaysia", "Singapore"]:

        subset = data.loc[
            (data["Configuration"] == configuration)
            & (data["Country"] == country)
        ].copy()

        train = subset.loc[
            subset["Split"] == "Train_2015_2022"
        ].copy()

        validation = subset.loc[
            subset["Split"] == "Validation_2023_2024"
        ].copy()

        X_train = train[feature_columns]
        X_valid = validation[feature_columns]

        # Convert labels from -1, 0, 1 to 0, 1, 2
        y_train = train["Label"].to_numpy() + 1
        y_valid = validation["Label"].to_numpy() + 1

        assert set(np.unique(y_train)) == {0, 1, 2}
        assert len(train) > 0
        assert len(validation) > 0

        # Train XGBoost without using validation data for fitting
        model = XGBClassifier(**model_params)

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        predictions = model.predict(X_valid)

        # Determine the majority class using training labels only
        majority_class = int(
            pd.Series(y_train).value_counts().idxmax()
        )

        baseline_predictions = np.full(
            len(y_valid),
            majority_class,
            dtype=int
        )

        # Evaluate XGBoost and majority-class baseline
        result = {
            "Configuration": configuration,
            "Country": country,
            "Train_Rows": len(train),
            "Validation_Rows": len(validation),
            "Accuracy": accuracy_score(
                y_valid, predictions
            ),
            "Macro_F1": f1_score(
                y_valid,
                predictions,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_valid, predictions
            ),
            "MCC": matthews_corrcoef(
                y_valid, predictions
            ),
            "Baseline_Class": majority_class - 1,
            "Baseline_Accuracy": accuracy_score(
                y_valid, baseline_predictions
            ),
            "Baseline_Macro_F1": f1_score(
                y_valid,
                baseline_predictions,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0
            ),
            "Baseline_MCC": matthews_corrcoef(
                y_valid, baseline_predictions
            )
        }

        result["F1_Improvement"] = (
            result["Macro_F1"] - result["Baseline_Macro_F1"]
        )

        results.append(result)

        # Save validation predictions for later analysis
        predictions_df = validation[
            [
                "Ticker",
                "Country",
                "Date",
                "Event_End_Date",
                "Label"
            ]
        ].copy()

        predictions_df["Configuration"] = configuration
        predictions_df["Predicted_Label"] = predictions - 1
        predictions_df["Baseline_Label"] = majority_class - 1

        prediction_records.append(predictions_df)

        trained_models[(configuration, country)] = model

# Combine evaluation results
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    ["Country", "Macro_F1"],
    ascending=[True, False]
).reset_index(drop=True)

validation_predictions = pd.concat(
    prediction_records,
    ignore_index=True
)

# Save evaluation results
results_df.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_initial_results.csv",
    index=False
)

validation_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_validation_predictions.csv",
    index=False
)

print("=== INITIAL XGBOOST VALIDATION RESULTS ===")

display(
    results_df[
        [
            "Configuration",
            "Country",
            "Train_Rows",
            "Validation_Rows",
            "Accuracy",
            "Macro_F1",
            "Balanced_Accuracy",
            "MCC",
            "Baseline_Accuracy",
            "Baseline_Macro_F1",
            "F1_Improvement"
        ]
    ].round(4)
)

print("\n=== BEST INITIAL CONFIGURATION BY MARKET ===")

best_configs = (
    results_df
    .sort_values("Macro_F1", ascending=False)
    .groupby("Country")
    .head(1)
    .sort_values("Country")
)

display(
    best_configs[
        [
            "Country",
            "Configuration",
            "Macro_F1",
            "Baseline_Macro_F1",
            "Balanced_Accuracy",
            "MCC"
        ]
    ].round(4)
)

print("\nCompleted.")

=== INITIAL XGBOOST VALIDATION RESULTS ===


,Configuration,Country,Train_Rows,Validation_Rows,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Baseline_Accuracy,Baseline_Macro_F1,F1_Improvement
0,V5_H10_M2,Malaysia,18427,4802,0.4398,0.4406,0.4570,0.1715,0.3290,0.1650,0.2756
1,V5_H5_M2,Malaysia,18519,4821,0.5401,0.4403,0.4674,0.2553,0.4798,0.2161,0.2242
2,V10_H10_M2,Malaysia,18434,4811,0.4070,0.4031,0.4143,0.1148,0.3224,0.1625,0.2405
3,V10_H5_M2,Malaysia,18531,4829,0.5421,0.3715,0.3987,0.1561,0.5291,0.2307,0.1409
4,V5_H5_M2,Singapore,5908,1488,0.4866,0.4440,0.4465,0.1945,0.4422,0.2044,0.2396
5,V5_H10_M2,Singapore,5897,1479,0.4300,0.4059,0.3955,0.0694,0.4523,0.2076,0.1983
6,V10_H5_M2,Singapore,5906,1488,0.4724,0.3956,0.4007,0.1291,0.4812,0.2166,0.1790
7,V10_H10_M2,Singapore,5895,1481,0.4045,0.3609,0.3620,0.0333,0.4463,0.2057,0.1552



=== BEST INITIAL CONFIGURATION BY MARKET ===


,Country,Configuration,Macro_F1,Baseline_Macro_F1,Balanced_Accuracy,MCC
0,Malaysia,V5_H10_M2,0.4406,0.1650,0.4570,0.1715
4,Singapore,V5_H5_M2,0.4440,0.2044,0.4465,0.1945



Completed.


**Examine per-class validation performance**

In [2]:
from sklearn.metrics import confusion_matrix, classification_report

# Select the top two configurations per market
selected_results = (
    results_df
    .sort_values(
        ["Country", "Macro_F1"],
        ascending=[True, False]
    )
    .groupby("Country")
    .head(2)
)

class_names = ["Down", "Flat", "Up"]
class_labels = [-1, 0, 1]

class_records = []
distribution_records = []
confusion_results = {}

for _, result in selected_results.iterrows():

    country = result["Country"]
    configuration = result["Configuration"]

    subset = validation_predictions.loc[
        (validation_predictions["Country"] == country)
        & (validation_predictions["Configuration"] == configuration)
    ].copy()

    y_true = subset["Label"].to_numpy()
    y_pred = subset["Predicted_Label"].to_numpy()

    # Calculate confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_labels
    )

    confusion_results[(country, configuration)] = cm

    # Calculate per-class metrics
    report = classification_report(
        y_true,
        y_pred,
        labels=class_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    for name in class_names:
        class_records.append({
            "Country": country,
            "Configuration": configuration,
            "Class": name,
            "Precision": report[name]["precision"],
            "Recall": report[name]["recall"],
            "F1": report[name]["f1-score"],
            "Support": int(report[name]["support"])
        })

    # Compare actual and predicted class frequencies
    for label, name in zip(class_labels, class_names):
        distribution_records.append({
            "Country": country,
            "Configuration": configuration,
            "Class": name,
            "Actual_Count": int((y_true == label).sum()),
            "Predicted_Count": int((y_pred == label).sum()),
            "Actual_Pct": round(
                (y_true == label).mean() * 100, 2
            ),
            "Predicted_Pct": round(
                (y_pred == label).mean() * 100, 2
            )
        })

class_metrics = pd.DataFrame(class_records)
prediction_distribution = pd.DataFrame(distribution_records)

print("=== PER-CLASS VALIDATION METRICS ===")

display(
    class_metrics.round(4)
)

print("\n=== ACTUAL VS PREDICTED CLASS DISTRIBUTIONS ===")

display(
    prediction_distribution
)

print("\n=== CONFUSION MATRICES ===")

for (country, configuration), cm in confusion_results.items():

    print(f"\n{country} — {configuration}")
    print("Rows: Actual class")
    print("Columns: Predicted class")

    cm_df = pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names
    )

    display(cm_df)

# Save diagnostic results
class_metrics.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_class_metrics.csv",
    index=False
)

prediction_distribution.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_prediction_distribution.csv",
    index=False
)

print("\nCompleted.")

=== PER-CLASS VALIDATION METRICS ===


,Country,Configuration,Class,Precision,Recall,F1,Support
0,Malaysia,V5_H10_M2,Down,0.3839,0.6418,0.4805,1580
1,Malaysia,V5_H10_M2,Flat,0.5571,0.4716,0.5108,1251
2,Malaysia,V5_H10_M2,Up,0.4610,0.2577,0.3306,1971
3,Malaysia,V5_H5_M2,Down,0.3729,0.4441,0.4054,1153
4,Malaysia,V5_H5_M2,Flat,0.6307,0.8284,0.7161,2313
5,Malaysia,V5_H5_M2,Up,0.4293,0.1299,0.1994,1355
6,Singapore,V5_H5_M2,Down,0.3420,0.3140,0.3274,379
7,Singapore,V5_H5_M2,Flat,0.5891,0.6884,0.6349,658
8,Singapore,V5_H5_M2,Up,0.4097,0.3370,0.3698,451
9,Singapore,V5_H10_M2,Down,0.3770,0.4487,0.4097,526



=== ACTUAL VS PREDICTED CLASS DISTRIBUTIONS ===


,Country,Configuration,Class,Actual_Count,Predicted_Count,Actual_Pct,Predicted_Pct
0,Malaysia,V5_H10_M2,Down,1580,2641,32.90,55.00
1,Malaysia,V5_H10_M2,Flat,1251,1059,26.05,22.05
2,Malaysia,V5_H10_M2,Up,1971,1102,41.05,22.95
3,Malaysia,V5_H5_M2,Down,1153,1373,23.92,28.48
4,Malaysia,V5_H5_M2,Flat,2313,3038,47.98,63.02
5,Malaysia,V5_H5_M2,Up,1355,410,28.11,8.50
6,Singapore,V5_H5_M2,Down,379,348,25.47,23.39
7,Singapore,V5_H5_M2,Flat,658,769,44.22,51.68
8,Singapore,V5_H5_M2,Up,451,371,30.31,24.93
9,Singapore,V5_H10_M2,Down,526,626,35.56,42.33



=== CONFUSION MATRICES ===

Malaysia — V5_H10_M2
Rows: Actual class
Columns: Predicted class


,Down,Flat,Up
Down,1014,192,374
Flat,441,590,220
Up,1186,277,508


... (3 outputs left out)

,Down,Flat,Up
Down,119,149,111
Flat,97,453,108
Up,132,167,152



Singapore — V5_H10_M2
Rows: Actual class
Columns: Predicted class


,Down,Flat,Up
Down,236,29,261
Flat,86,69,129
Up,304,34,331



Completed.


**Analyse validation performance by year and bank**

In [3]:
# Select the top two configurations for each market
selected_pairs = set(
    zip(
        selected_results["Country"],
        selected_results["Configuration"]
    )
)

diagnostic_data = validation_predictions.loc[
    validation_predictions.apply(
        lambda row: (
            row["Country"],
            row["Configuration"]
        ) in selected_pairs,
        axis=1
    )
].copy()

# Extract validation year
diagnostic_data["Year"] = diagnostic_data["Date"].dt.year

# Calculate class recall
def calculate_class_recall(y_true, y_pred, target_class):
    mask = y_true == target_class

    if mask.sum() == 0:
        return np.nan

    return np.mean(y_pred[mask] == target_class)


# Calculate evaluation metrics for each group
def evaluate_group(group):
    y_true = group["Label"].to_numpy()
    y_pred = group["Predicted_Label"].to_numpy()
    y_baseline = group["Baseline_Label"].to_numpy()

    return pd.Series({
        "Observations": len(group),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Down_Recall": calculate_class_recall(
            y_true, y_pred, -1
        ),
        "Flat_Recall": calculate_class_recall(
            y_true, y_pred, 0
        ),
        "Up_Recall": calculate_class_recall(
            y_true, y_pred, 1
        ),
        "Baseline_Macro_F1": f1_score(
            y_true,
            y_baseline,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )
    })


# Evaluate performance separately for 2023 and 2024
yearly_results = (
    diagnostic_data
    .groupby(
        ["Country", "Configuration", "Year"]
    )
    .apply(evaluate_group, include_groups=False)
    .reset_index()
)

print("=== YEARLY VALIDATION PERFORMANCE ===")

display(
    yearly_results.round(4)
)

# Evaluate performance for each banking stock
bank_results = (
    diagnostic_data
    .groupby(
        ["Country", "Configuration", "Ticker"]
    )
    .apply(evaluate_group, include_groups=False)
    .reset_index()
)

print("\n=== BANK-LEVEL VALIDATION PERFORMANCE ===")

display(
    bank_results[
        [
            "Country",
            "Configuration",
            "Ticker",
            "Observations",
            "Accuracy",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Identify the weakest Up recall in each market
weakest_up = (
    bank_results
    .sort_values("Up_Recall")
    .groupby(["Country", "Configuration"])
    .head(3)
)

print("\n=== LOWEST UP RECALL BY MARKET AND CONFIGURATION ===")

display(
    weakest_up[
        [
            "Country",
            "Configuration",
            "Ticker",
            "Observations",
            "Up_Recall",
            "Macro_F1"
        ]
    ].round(4)
)

# Save diagnostic reports
yearly_results.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_yearly_performance.csv",
    index=False
)

bank_results.to_csv(
    RESULTS_DIR / "triple_barrier_xgboost_bank_performance.csv",
    index=False
)

print("\nCompleted.")

=== YEARLY VALIDATION PERFORMANCE ===


,Country,Configuration,Year,Observations,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall,Baseline_Macro_F1
0,Malaysia,V5_H10_M2,2023,2435.0,0.4456,0.4399,0.4501,0.1649,0.6609,0.4392,0.2503,0.1750
1,Malaysia,V5_H10_M2,2024,2367.0,0.4339,0.4395,0.4626,0.1771,0.6185,0.5049,0.2645,0.1543
2,Malaysia,V5_H5_M2,2023,2435.0,0.5351,0.4371,0.4618,0.2462,0.4640,0.7998,0.1217,0.2162
3,Malaysia,V5_H5_M2,2024,2386.0,0.5453,0.4421,0.4717,0.2639,0.4205,0.8575,0.1373,0.2161
4,Singapore,V5_H10_M2,2023,747.0,0.4177,0.3774,0.3807,0.0601,0.3470,0.1818,0.6132,0.1850
5,Singapore,V5_H10_M2,2024,732.0,0.4426,0.4321,0.4379,0.1302,0.6029,0.3050,0.4058,0.2286
6,Singapore,V5_H5_M2,2023,747.0,0.4485,0.4074,0.4152,0.1442,0.2424,0.6698,0.3333,0.1991
7,Singapore,V5_H5_M2,2024,741.0,0.5250,0.4817,0.4905,0.2555,0.4257,0.7059,0.3399,0.2097



=== BANK-LEVEL VALIDATION PERFORMANCE ===


,Country,Configuration,Ticker,Observations,Accuracy,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,V5_H10_M2,1015.KL,485.0,0.4206,0.4254,0.1506,0.6014,0.4836,0.2605
1,Malaysia,V5_H10_M2,1023.KL,488.0,0.4160,0.4216,0.1021,0.5941,0.4750,0.2689
2,Malaysia,V5_H10_M2,1066.KL,485.0,0.4021,0.3995,0.1567,0.7313,0.3444,0.2250
3,Malaysia,V5_H10_M2,1155.KL,485.0,0.4082,0.4053,0.1277,0.7070,0.3778,0.1865
4,Malaysia,V5_H10_M2,1171.KL,484.0,0.4339,0.4354,0.1545,0.5389,0.5469,0.2646
5,Malaysia,V5_H10_M2,1295.KL,439.0,0.4875,0.4801,0.2034,0.6761,0.5000,0.2938
6,Malaysia,V5_H10_M2,2488.KL,485.0,0.4680,0.4694,0.2093,0.6000,0.5167,0.3366
7,Malaysia,V5_H10_M2,5185.KL,486.0,0.4897,0.4888,0.2487,0.6013,0.5844,0.3128
8,Malaysia,V5_H10_M2,5258.KL,480.0,0.4021,0.4016,0.1113,0.5309,0.5111,0.2077
9,Malaysia,V5_H10_M2,5819.KL,485.0,0.4742,0.4568,0.2661,0.8627,0.4000,0.2188



=== LOWEST UP RECALL BY MARKET AND CONFIGURATION ===


,Country,Configuration,Ticker,Observations,Up_Recall,Macro_F1
12,Malaysia,V5_H5_M2,1066.KL,486.0,0.0692,0.4203
10,Malaysia,V5_H5_M2,1015.KL,486.0,0.0884,0.3859
18,Malaysia,V5_H5_M2,5258.KL,485.0,0.1032,0.4098
3,Malaysia,V5_H10_M2,1155.KL,485.0,0.1865,0.4053
8,Malaysia,V5_H10_M2,5258.KL,480.0,0.2077,0.4016
9,Malaysia,V5_H10_M2,5819.KL,485.0,0.2188,0.4568
23,Singapore,V5_H5_M2,D05.SI,496.0,0.2400,0.3878
24,Singapore,V5_H5_M2,O39.SI,496.0,0.3067,0.4679
25,Singapore,V5_H5_M2,U11.SI,496.0,0.4636,0.4681
21,Singapore,V5_H10_M2,O39.SI,493.0,0.4846,0.4168



Completed.


**Prepare weekly walk-forward training schedule**

In [4]:
# Select candidate Triple Barrier configurations
selected_configurations = [
    "V5_H5_M2",
    "V5_H10_M2"
]

walkforward_data = data.loc[
    data["Configuration"].isin(selected_configurations)
].copy()

# Calculate the Monday of each prediction week
walkforward_data["Week_Start"] = (
    walkforward_data["Date"]
    - pd.to_timedelta(
        walkforward_data["Date"].dt.weekday,
        unit="D"
    )
)

# Define validation dates
VALIDATION_START = pd.Timestamp("2023-01-01")
VALIDATION_END = pd.Timestamp("2024-12-31")

# Select validation observations
walkforward_validation = walkforward_data.loc[
    (walkforward_data["Split"] == "Validation_2023_2024")
    & (walkforward_data["Date"] >= VALIDATION_START)
    & (walkforward_data["Date"] <= VALIDATION_END)
].copy()

# Define training history
walkforward_history = walkforward_data.loc[
    walkforward_data["Split"] == "Train_2015_2022"
].copy()

# Store the available historical training observations
schedule_records = []

for configuration in selected_configurations:

    for country in ["Malaysia", "Singapore"]:

        history = walkforward_history.loc[
            (walkforward_history["Configuration"] == configuration)
            & (walkforward_history["Country"] == country)
        ]

        validation = walkforward_validation.loc[
            (walkforward_validation["Configuration"] == configuration)
            & (walkforward_validation["Country"] == country)
        ]

        for week_start, week_data in validation.groupby("Week_Start"):

            # Require all training labels to mature before retraining
            eligible_history = history.loc[
                (history["Date"] < week_start)
                & (history["Event_End_Date"] < week_start)
            ]

            # Define the 104-week rolling training window
            rolling_start = (
                week_start - pd.Timedelta(weeks=104)
            )

            rolling_history = eligible_history.loc[
                eligible_history["Date"] >= rolling_start
            ]

            schedule_records.append({
                "Configuration": configuration,
                "Country": country,
                "Week_Start": week_start,
                "Prediction_Rows": len(week_data),
                "First_Prediction_Date": week_data["Date"].min(),
                "Last_Prediction_Date": week_data["Date"].max(),
                "Expanding_Train_Rows": len(eligible_history),
                "Rolling_104W_Train_Rows": len(rolling_history),
                "Latest_Training_Event_End": (
                    eligible_history["Event_End_Date"].max()
                )
            })

walkforward_schedule = pd.DataFrame(schedule_records)

# Validate training-label maturity
assert (
    walkforward_schedule["Latest_Training_Event_End"]
    < walkforward_schedule["Week_Start"]
).all()

assert (
    walkforward_schedule["Prediction_Rows"] > 0
).all()

# Summarise weekly schedule
schedule_summary = (
    walkforward_schedule
    .groupby(["Country", "Configuration"])
    .agg(
        Prediction_Weeks=("Week_Start", "nunique"),
        Prediction_Rows=("Prediction_Rows", "sum"),
        First_Week=("Week_Start", "min"),
        Last_Week=("Week_Start", "max"),
        Min_Expanding_Train=("Expanding_Train_Rows", "min"),
        Max_Expanding_Train=("Expanding_Train_Rows", "max"),
        Min_Rolling_104W_Train=("Rolling_104W_Train_Rows", "min")
    )
    .reset_index()
)

print("=== WEEKLY WALK-FORWARD SCHEDULE ===")
display(schedule_summary)

print("\n=== FIRST 12 TRAINING WEEKS ===")

display(
    walkforward_schedule.sort_values(
        ["Country", "Configuration", "Week_Start"]
    ).head(12)
)

# Save walk-forward schedule
walkforward_schedule.to_csv(
    RESULTS_DIR / "triple_barrier_walkforward_schedule.csv",
    index=False
)

print("\nCompleted.")

=== WEEKLY WALK-FORWARD SCHEDULE ===


,Country,Configuration,Prediction_Weeks,Prediction_Rows,First_Week,Last_Week,Min_Expanding_Train,Max_Expanding_Train,Min_Rolling_104W_Train
0,Malaysia,V5_H10_M2,105,4802,2023-01-02,2024-12-30,18427,18427,0
1,Malaysia,V5_H5_M2,105,4821,2023-01-02,2024-12-30,18519,18519,0
2,Singapore,V5_H10_M2,103,1479,2023-01-02,2024-12-16,5897,5897,7
3,Singapore,V5_H5_M2,104,1488,2023-01-02,2024-12-23,5908,5908,3



=== FIRST 12 TRAINING WEEKS ===


,Configuration,Country,Week_Start,Prediction_Rows,First_Prediction_Date,Last_Prediction_Date,Expanding_Train_Rows,Rolling_104W_Train_Rows,Latest_Training_Event_End
209,V5_H10_M2,Malaysia,2023-01-02,40,2023-01-03,2023-01-06,18427,4682,2022-12-30
210,V5_H10_M2,Malaysia,2023-01-09,50,2023-01-09,2023-01-13,18427,4632,2022-12-30
211,V5_H10_M2,Malaysia,2023-01-16,50,2023-01-16,2023-01-20,18427,4582,2022-12-30
212,V5_H10_M2,Malaysia,2023-01-23,30,2023-01-25,2023-01-27,18427,4532,2022-12-30
213,V5_H10_M2,Malaysia,2023-01-30,40,2023-01-30,2023-02-03,18427,4492,2022-12-30
214,V5_H10_M2,Malaysia,2023-02-06,40,2023-02-07,2023-02-10,18427,4452,2022-12-30
215,V5_H10_M2,Malaysia,2023-02-13,48,2023-02-13,2023-02-17,18427,4412,2022-12-30
216,V5_H10_M2,Malaysia,2023-02-20,50,2023-02-20,2023-02-24,18427,4367,2022-12-30
217,V5_H10_M2,Malaysia,2023-02-27,50,2023-02-27,2023-03-03,18427,4322,2022-12-30
218,V5_H10_M2,Malaysia,2023-03-06,50,2023-03-06,2023-03-10,18427,4277,2022-12-30



Completed.


**Correct walk-forward training history**

In [5]:
# Define candidate configurations and training windows
selected_configurations = [
    "V5_H5_M2",
    "V5_H10_M2"
]

ROLLING_WEEKS = 104

# Retain all development observations for selected configurations
walkforward_data = data.loc[
    data["Configuration"].isin(selected_configurations)
].copy()

# Calculate the Monday of each observation week
walkforward_data["Week_Start"] = (
    walkforward_data["Date"]
    - pd.to_timedelta(
        walkforward_data["Date"].dt.weekday,
        unit="D"
    )
)

# Identify validation observations
walkforward_validation = walkforward_data.loc[
    walkforward_data["Split"] == "Validation_2023_2024"
].copy()

schedule_records = []

for configuration in selected_configurations:

    for country in ["Malaysia", "Singapore"]:

        # Select all available development history
        history = walkforward_data.loc[
            (walkforward_data["Configuration"] == configuration)
            & (walkforward_data["Country"] == country)
        ].copy()

        # Select validation observations
        validation = walkforward_validation.loc[
            (walkforward_validation["Configuration"] == configuration)
            & (walkforward_validation["Country"] == country)
        ].copy()

        for week_start, week_data in validation.groupby("Week_Start"):

            # Include only labels known before the prediction week
            eligible_history = history.loc[
                (history["Date"] < week_start)
                & (history["Event_End_Date"] < week_start)
            ]

            # Restrict rolling training to the previous 104 weeks
            rolling_start = (
                week_start - pd.Timedelta(weeks=ROLLING_WEEKS)
            )

            rolling_history = eligible_history.loc[
                eligible_history["Date"] >= rolling_start
            ]

            # Verify label maturity and training dates
            assert (
                eligible_history["Event_End_Date"] < week_start
            ).all()

            assert (
                rolling_history["Date"] >= rolling_start
            ).all()

            assert (
                rolling_history["Event_End_Date"] < week_start
            ).all()

            schedule_records.append({
                "Configuration": configuration,
                "Country": country,
                "Week_Start": week_start,
                "Prediction_Rows": len(week_data),
                "First_Prediction_Date": week_data["Date"].min(),
                "Last_Prediction_Date": week_data["Date"].max(),
                "Expanding_Train_Rows": len(eligible_history),
                "Rolling_104W_Train_Rows": len(rolling_history),
                "Training_Rows_From_2023": int(
                    (eligible_history["Date"] >= "2023-01-01").sum()
                ),
                "Latest_Training_Event_End": (
                    eligible_history["Event_End_Date"].max()
                )
            })

# Assemble corrected schedule
walkforward_schedule = pd.DataFrame(schedule_records)

walkforward_schedule = (
    walkforward_schedule
    .sort_values(["Country", "Configuration", "Week_Start"])
    .reset_index(drop=True)
)

# Validate schedule
assert (
    walkforward_schedule["Latest_Training_Event_End"]
    < walkforward_schedule["Week_Start"]
).all()

assert (
    walkforward_schedule["Rolling_104W_Train_Rows"] > 0
).all()

assert (
    walkforward_schedule["Rolling_104W_Train_Rows"]
    <= walkforward_schedule["Expanding_Train_Rows"]
).all()

assert (
    walkforward_schedule["Training_Rows_From_2023"].max() > 0
)

# Summarise available training history
schedule_summary = (
    walkforward_schedule
    .groupby(["Country", "Configuration"])
    .agg(
        Prediction_Weeks=("Week_Start", "nunique"),
        Prediction_Rows=("Prediction_Rows", "sum"),
        First_Week=("Week_Start", "min"),
        Last_Week=("Week_Start", "max"),
        Min_Expanding_Train=("Expanding_Train_Rows", "min"),
        Max_Expanding_Train=("Expanding_Train_Rows", "max"),
        Min_Rolling_104W_Train=("Rolling_104W_Train_Rows", "min"),
        Max_Rolling_104W_Train=("Rolling_104W_Train_Rows", "max"),
        Max_Training_Rows_From_2023=(
            "Training_Rows_From_2023", "max"
        )
    )
    .reset_index()
)

print("=== CORRECTED WALK-FORWARD SCHEDULE ===")
display(schedule_summary)

# Inspect selected weeks across the validation period
sample_weeks = pd.concat([
    walkforward_schedule.groupby(
        ["Country", "Configuration"]
    ).head(2),

    walkforward_schedule.groupby(
        ["Country", "Configuration"]
    ).tail(2)
])

print("\n=== TRAINING HISTORY AT SELECTED WEEKS ===")

display(
    sample_weeks[
        [
            "Country",
            "Configuration",
            "Week_Start",
            "Prediction_Rows",
            "Expanding_Train_Rows",
            "Rolling_104W_Train_Rows",
            "Training_Rows_From_2023",
            "Latest_Training_Event_End"
        ]
    ].sort_values(
        ["Country", "Configuration", "Week_Start"]
    )
)

# Save corrected schedule
walkforward_schedule.to_csv(
    RESULTS_DIR / "triple_barrier_walkforward_schedule.csv",
    index=False
)

print("\nCompleted.")

=== CORRECTED WALK-FORWARD SCHEDULE ===


,Country,Configuration,Prediction_Weeks,Prediction_Rows,First_Week,Last_Week,Min_Expanding_Train,Max_Expanding_Train,Min_Rolling_104W_Train,Max_Rolling_104W_Train,Max_Training_Rows_From_2023
0,Malaysia,V5_H10_M2,105,4802,2023-01-02,2024-12-30,18427,23215,4595,4788,4788
1,Malaysia,V5_H5_M2,105,4821,2023-01-02,2024-12-30,18519,23324,4647,4813,4805
2,Singapore,V5_H10_M2,103,1479,2023-01-02,2024-12-16,5897,7348,1445,1487,1451
3,Singapore,V5_H5_M2,104,1488,2023-01-02,2024-12-23,5908,7387,1468,1498,1479



=== TRAINING HISTORY AT SELECTED WEEKS ===


,Country,Configuration,Week_Start,Prediction_Rows,Expanding_Train_Rows,Rolling_104W_Train_Rows,Training_Rows_From_2023,Latest_Training_Event_End
0,Malaysia,V5_H10_M2,2023-01-02,40,18427,4682,0,2022-12-30
1,Malaysia,V5_H10_M2,2023-01-09,50,18428,4633,1,2023-01-06
103,Malaysia,V5_H10_M2,2024-12-23,9,23195,4776,4768,2024-12-20
104,Malaysia,V5_H10_M2,2024-12-30,1,23215,4788,4788,2024-12-27
105,Malaysia,V5_H5_M2,2023-01-02,40,18519,4714,0,2022-12-30
106,Malaysia,V5_H5_M2,2023-01-09,50,18520,4665,1,2023-01-06
208,Malaysia,V5_H5_M2,2024-12-23,16,23296,4785,4777,2024-12-20
209,Malaysia,V5_H5_M2,2024-12-30,1,23324,4805,4805,2024-12-27
210,Singapore,V5_H10_M2,2023-01-02,12,5897,1487,0,2022-12-30
211,Singapore,V5_H10_M2,2023-01-09,15,5903,1478,6,2023-01-06



Completed.


**Train weekly expanding-history XGBoost models**

In [6]:
CONFIGURATION = "V5_H5_M2"

# Load complete Triple Barrier labels
all_labels = pd.read_csv(
    PROCESSED_DIR / "triple_barrier_configuration_labels.csv",
    parse_dates=["Date", "Event_End_Date"]
)

# Load the complete feature dataset
all_features = pd.read_csv(
    PROCESSED_DIR / "lseg_feature_engineered.csv",
    parse_dates=["Date"]
)

# Select the required configuration and historical period
all_labels = all_labels.loc[
    (all_labels["Configuration"] == CONFIGURATION)
    & (all_labels["Date"] < "2025-01-01")
].copy()

# Recover complete historical feature and label records
history_data = all_labels.merge(
    all_features[
        ["Ticker", "Date"] + feature_columns
    ],
    on=["Ticker", "Date"],
    how="inner",
    validate="many_to_one"
)

# Retain observations with complete model features
history_data = history_data.loc[
    history_data[feature_columns].notna().all(axis=1)
].copy()

history_data = history_data.sort_values(
    ["Country", "Date", "Ticker"]
).reset_index(drop=True)

# Retain the original validation observations for a fair comparison
validation_data = data.loc[
    (data["Configuration"] == CONFIGURATION)
    & (data["Split"] == "Validation_2023_2024")
].copy()

validation_data["Week_Start"] = (
    validation_data["Date"]
    - pd.to_timedelta(
        validation_data["Date"].dt.weekday,
        unit="D"
    )
)

# Verify that historical labels and validation labels agree
label_check = validation_data[
    ["Ticker", "Date", "Label"]
].merge(
    history_data[["Ticker", "Date", "Label"]],
    on=["Ticker", "Date"],
    how="left",
    suffixes=("_validation", "_history"),
    validate="one_to_one"
)

assert (
    label_check["Label_validation"]
    == label_check["Label_history"]
).all()

# Store weekly predictions and training diagnostics
walkforward_predictions = []
training_records = []

for country in ["Malaysia", "Singapore"]:

    country_history = history_data.loc[
        history_data["Country"] == country
    ].copy()

    country_validation = validation_data.loc[
        validation_data["Country"] == country
    ].copy()

    weekly_groups = list(
        country_validation.groupby("Week_Start", sort=True)
    )

    print(f"\n=== {country.upper()} ===")
    print("Prediction weeks:", len(weekly_groups))

    for week_number, (week_start, week_data) in enumerate(
        weekly_groups,
        start=1
    ):

        # Select only events known before the prediction week
        train = country_history.loc[
            (country_history["Date"] < week_start)
            & (country_history["Event_End_Date"] < week_start)
        ].copy()

        assert len(train) > 0
        assert train["Event_End_Date"].max() < week_start

        # Prepare training and prediction features
        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy() + 1

        X_valid = week_data[feature_columns]
        y_valid = week_data["Label"].to_numpy()

        assert set(np.unique(y_train)) == {0, 1, 2}
        assert X_train.notna().all().all()
        assert X_valid.notna().all().all()

        # Retrain XGBoost using all currently matured events
        model = XGBClassifier(**model_params)

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        predictions = model.predict(X_valid) - 1

        # Determine majority class using current training history
        majority_class = int(
            train["Label"].value_counts().idxmax()
        )

        # Store daily predictions from the weekly model
        week_predictions = week_data[
            [
                "Ticker",
                "Country",
                "Date",
                "Event_End_Date",
                "Label",
                "Week_Start"
            ]
        ].copy()

        week_predictions["Configuration"] = CONFIGURATION
        week_predictions["Predicted_Label"] = predictions
        week_predictions["Baseline_Label"] = majority_class
        week_predictions["Training_Strategy"] = "Expanding"

        walkforward_predictions.append(week_predictions)

        # Store training-history diagnostics
        training_records.append({
            "Country": country,
            "Configuration": CONFIGURATION,
            "Week_Start": week_start,
            "Training_Rows": len(train),
            "Prediction_Rows": len(week_data),
            "Latest_Training_Event_End": (
                train["Event_End_Date"].max()
            ),
            "Training_Rows_From_2023": int(
                (train["Date"] >= "2023-01-01").sum()
            )
        })

        # Display periodic progress
        if (
            week_number == 1
            or week_number % 20 == 0
            or week_number == len(weekly_groups)
        ):
            print(
                f"Week {week_number}/{len(weekly_groups)} | "
                f"Training rows: {len(train)} | "
                f"Prediction rows: {len(week_data)}"
            )

# Combine daily validation predictions
expanding_predictions = pd.concat(
    walkforward_predictions,
    ignore_index=True
)

expanding_training_log = pd.DataFrame(training_records)

# Verify every validation observation has exactly one prediction
assert not expanding_predictions.duplicated(
    ["Configuration", "Ticker", "Date"]
).any()

assert len(expanding_predictions) == len(validation_data)

assert (
    expanding_training_log["Latest_Training_Event_End"]
    < expanding_training_log["Week_Start"]
).all()

# Evaluate performance by market
expanding_results = []

for country, subset in expanding_predictions.groupby("Country"):

    y_true = subset["Label"].to_numpy()
    y_pred = subset["Predicted_Label"].to_numpy()
    y_baseline = subset["Baseline_Label"].to_numpy()

    expanding_results.append({
        "Country": country,
        "Configuration": CONFIGURATION,
        "Strategy": "Weekly_Expanding",
        "Validation_Rows": len(subset),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true, y_pred,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Down_Recall": np.mean(
            y_pred[y_true == -1] == -1
        ),
        "Flat_Recall": np.mean(
            y_pred[y_true == 0] == 0
        ),
        "Up_Recall": np.mean(
            y_pred[y_true == 1] == 1
        ),
        "Baseline_Accuracy": accuracy_score(
            y_true, y_baseline
        ),
        "Baseline_Macro_F1": f1_score(
            y_true, y_baseline,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )
    })

expanding_results_df = pd.DataFrame(expanding_results)

# Compare walk-forward results with fixed-training results
fixed_results = results_df.loc[
    results_df["Configuration"] == CONFIGURATION,
    ["Country", "Accuracy", "Macro_F1", "MCC"]
].copy()

fixed_results = fixed_results.rename(columns={
    "Accuracy": "Fixed_Accuracy",
    "Macro_F1": "Fixed_Macro_F1",
    "MCC": "Fixed_MCC"
})

comparison = expanding_results_df.merge(
    fixed_results,
    on="Country",
    how="left",
    validate="one_to_one"
)

comparison["Macro_F1_Change"] = (
    comparison["Macro_F1"] - comparison["Fixed_Macro_F1"]
)

comparison["MCC_Change"] = (
    comparison["MCC"] - comparison["Fixed_MCC"]
)

print("\n=== EXPANDING WALK-FORWARD RESULTS ===")

display(expanding_results_df.round(4))

print("\n=== FIXED VS EXPANDING COMPARISON ===")

display(
    comparison[
        [
            "Country",
            "Validation_Rows",
            "Fixed_Accuracy",
            "Accuracy",
            "Fixed_Macro_F1",
            "Macro_F1",
            "Macro_F1_Change",
            "Fixed_MCC",
            "MCC",
            "MCC_Change"
        ]
    ].round(4)
)

print("\n=== TRAINING HISTORY SUMMARY ===")

display(
    expanding_training_log.groupby("Country").agg(
        Weeks=("Week_Start", "nunique"),
        Min_Train_Rows=("Training_Rows", "min"),
        Max_Train_Rows=("Training_Rows", "max"),
        Max_2023_Rows=("Training_Rows_From_2023", "max")
    )
)

# Save predictions, evaluation results and training logs
expanding_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_expanding_predictions.csv",
    index=False
)

expanding_results_df.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_expanding_results.csv",
    index=False
)

expanding_training_log.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_expanding_training_log.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Prediction weeks: 105
Week 1/105 | Training rows: 18519 | Prediction rows: 40
Week 20/105 | Training rows: 19374 | Prediction rows: 50
Week 40/105 | Training rows: 20334 | Prediction rows: 50
Week 60/105 | Training rows: 21264 | Prediction rows: 50
Week 80/105 | Training rows: 22192 | Prediction rows: 40
Week 100/105 | Training rows: 23125 | Prediction rows: 47
Week 105/105 | Training rows: 23359 | Prediction rows: 1

=== SINGAPORE ===
Prediction weeks: 104
Week 1/104 | Training rows: 5908 | Prediction rows: 12
Week 20/104 | Training rows: 6179 | Prediction rows: 15
Week 40/104 | Training rows: 6462 | Prediction rows: 15
Week 60/104 | Training rows: 6760 | Prediction rows: 15
Week 80/104 | Training rows: 7040 | Prediction rows: 15
Week 100/104 | Training rows: 7329 | Prediction rows: 15
Week 104/104 | Training rows: 7398 | Prediction rows: 3

=== EXPANDING WALK-FORWARD RESULTS ===


,Country,Configuration,Strategy,Validation_Rows,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall,Baseline_Accuracy,Baseline_Macro_F1
0,Malaysia,V5_H5_M2,Weekly_Expanding,4821,0.5414,0.4451,0.4669,0.2531,0.4189,0.8314,0.1506,0.4798,0.2161
1,Singapore,V5_H5_M2,Weekly_Expanding,1488,0.4819,0.4337,0.4382,0.1854,0.3087,0.7067,0.2993,0.4422,0.2044



=== FIXED VS EXPANDING COMPARISON ===


,Country,Validation_Rows,Fixed_Accuracy,Accuracy,Fixed_Macro_F1,Macro_F1,Macro_F1_Change,Fixed_MCC,MCC,MCC_Change
0,Malaysia,4821,0.5401,0.5414,0.4403,0.4451,0.0047,0.2553,0.2531,-0.0022
1,Singapore,1488,0.4866,0.4819,0.4440,0.4337,-0.0103,0.1945,0.1854,-0.0091



=== TRAINING HISTORY SUMMARY ===


,Weeks,Min_Train_Rows,Max_Train_Rows,Max_2023_Rows
Country,,,,
Malaysia,105,18519,23359,4805
Singapore,104,5908,7398,1479



Completed.


**Train weekly rolling 104-week XGBoost models**

In [7]:
ROLLING_WEEKS = 104

rolling_prediction_records = []
rolling_training_records = []

for country in ["Malaysia", "Singapore"]:

    country_history = history_data.loc[
        history_data["Country"] == country
    ].copy()

    country_validation = validation_data.loc[
        validation_data["Country"] == country
    ].copy()

    weekly_groups = list(
        country_validation.groupby("Week_Start", sort=True)
    )

    print(f"\n=== {country.upper()} ===")
    print("Prediction weeks:", len(weekly_groups))

    for week_number, (week_start, week_data) in enumerate(
        weekly_groups,
        start=1
    ):

        # Define the rolling training period
        rolling_start = (
            week_start - pd.Timedelta(weeks=ROLLING_WEEKS)
        )

        # Select recent events with matured labels
        train = country_history.loc[
            (country_history["Date"] >= rolling_start)
            & (country_history["Date"] < week_start)
            & (country_history["Event_End_Date"] < week_start)
        ].copy()

        # Validate chronological training boundaries
        assert len(train) > 0
        assert train["Date"].min() >= rolling_start
        assert train["Date"].max() < week_start
        assert train["Event_End_Date"].max() < week_start

        # Prepare training and prediction data
        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy() + 1

        X_valid = week_data[feature_columns]

        assert set(np.unique(y_train)) == {0, 1, 2}
        assert X_train.notna().all().all()
        assert X_valid.notna().all().all()

        # Retrain XGBoost using the rolling history
        model = XGBClassifier(**model_params)

        model.fit(
            X_train,
            y_train,
            verbose=False
        )

        predictions = model.predict(X_valid) - 1

        # Calculate the current training majority class
        majority_class = int(
            train["Label"].value_counts().idxmax()
        )

        # Store daily predictions
        week_predictions = week_data[
            [
                "Ticker",
                "Country",
                "Date",
                "Event_End_Date",
                "Label",
                "Week_Start"
            ]
        ].copy()

        week_predictions["Configuration"] = CONFIGURATION
        week_predictions["Predicted_Label"] = predictions
        week_predictions["Baseline_Label"] = majority_class
        week_predictions["Training_Strategy"] = "Rolling_104W"

        rolling_prediction_records.append(week_predictions)

        # Record training-window information
        rolling_training_records.append({
            "Country": country,
            "Configuration": CONFIGURATION,
            "Week_Start": week_start,
            "Rolling_Start": rolling_start,
            "Training_Rows": len(train),
            "Prediction_Rows": len(week_data),
            "Earliest_Training_Date": train["Date"].min(),
            "Latest_Training_Event_End": (
                train["Event_End_Date"].max()
            )
        })

        if (
            week_number == 1
            or week_number % 20 == 0
            or week_number == len(weekly_groups)
        ):
            print(
                f"Week {week_number}/{len(weekly_groups)} | "
                f"Training rows: {len(train)} | "
                f"Prediction rows: {len(week_data)}"
            )

# Combine predictions and training logs
rolling_predictions = pd.concat(
    rolling_prediction_records,
    ignore_index=True
)

rolling_training_log = pd.DataFrame(
    rolling_training_records
)

# Validate prediction coverage
assert len(rolling_predictions) == len(validation_data)

assert not rolling_predictions.duplicated(
    ["Configuration", "Ticker", "Date"]
).any()

assert (
    rolling_training_log["Earliest_Training_Date"]
    >= rolling_training_log["Rolling_Start"]
).all()

assert (
    rolling_training_log["Latest_Training_Event_End"]
    < rolling_training_log["Week_Start"]
).all()

# Confirm the same observations were evaluated
rolling_keys = rolling_predictions[
    ["Ticker", "Date", "Label"]
].sort_values(["Ticker", "Date"]).reset_index(drop=True)

expanding_keys = expanding_predictions[
    ["Ticker", "Date", "Label"]
].sort_values(["Ticker", "Date"]).reset_index(drop=True)

assert rolling_keys.equals(expanding_keys)

# Evaluate rolling-window predictions
rolling_results = []

for country, subset in rolling_predictions.groupby("Country"):

    y_true = subset["Label"].to_numpy()
    y_pred = subset["Predicted_Label"].to_numpy()
    y_baseline = subset["Baseline_Label"].to_numpy()

    rolling_results.append({
        "Country": country,
        "Configuration": CONFIGURATION,
        "Strategy": "Weekly_Rolling_104W",
        "Validation_Rows": len(subset),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Down_Recall": np.mean(
            y_pred[y_true == -1] == -1
        ),
        "Flat_Recall": np.mean(
            y_pred[y_true == 0] == 0
        ),
        "Up_Recall": np.mean(
            y_pred[y_true == 1] == 1
        ),
        "Baseline_Accuracy": accuracy_score(
            y_true,
            y_baseline
        ),
        "Baseline_Macro_F1": f1_score(
            y_true,
            y_baseline,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )
    })

rolling_results_df = pd.DataFrame(rolling_results)

print("\n=== ROLLING 104-WEEK VALIDATION RESULTS ===")

display(rolling_results_df.round(4))

# Compare fixed, expanding and rolling training
comparison_records = []

for country in ["Malaysia", "Singapore"]:

    fixed = results_df.loc[
        (results_df["Country"] == country)
        & (results_df["Configuration"] == CONFIGURATION)
    ].iloc[0]

    expanding = expanding_results_df.loc[
        expanding_results_df["Country"] == country
    ].iloc[0]

    rolling = rolling_results_df.loc[
        rolling_results_df["Country"] == country
    ].iloc[0]

    for strategy, result in [
        ("Fixed", fixed),
        ("Weekly_Expanding", expanding),
        ("Weekly_Rolling_104W", rolling)
    ]:
        comparison_records.append({
            "Country": country,
            "Strategy": strategy,
            "Accuracy": result["Accuracy"],
            "Macro_F1": result["Macro_F1"],
            "Balanced_Accuracy": result["Balanced_Accuracy"],
            "MCC": result["MCC"]
        })

strategy_comparison = pd.DataFrame(
    comparison_records
)

print("\n=== TRAINING STRATEGY COMPARISON ===")

display(
    strategy_comparison.round(4)
)

print("\n=== ROLLING TRAINING HISTORY SUMMARY ===")

display(
    rolling_training_log.groupby("Country").agg(
        Weeks=("Week_Start", "nunique"),
        Min_Training_Rows=("Training_Rows", "min"),
        Max_Training_Rows=("Training_Rows", "max"),
        First_Training_Date=("Earliest_Training_Date", "min"),
        Latest_Training_Label=("Latest_Training_Event_End", "max")
    )
)

# Save rolling-window results
rolling_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_rolling_104w_predictions.csv",
    index=False
)

rolling_results_df.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_rolling_104w_results.csv",
    index=False
)

rolling_training_log.to_csv(
    RESULTS_DIR / "triple_barrier_weekly_rolling_104w_training_log.csv",
    index=False
)

strategy_comparison.to_csv(
    RESULTS_DIR / "triple_barrier_training_strategy_comparison.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Prediction weeks: 105
Week 1/105 | Training rows: 4714 | Prediction rows: 40
Week 20/105 | Training rows: 4745 | Prediction rows: 50
Week 40/105 | Training rows: 4789 | Prediction rows: 50
Week 60/105 | Training rows: 4816 | Prediction rows: 50
Week 80/105 | Training rows: 4804 | Prediction rows: 40
Week 100/105 | Training rows: 4801 | Prediction rows: 47
Week 105/105 | Training rows: 4805 | Prediction rows: 1

=== SINGAPORE ===
Prediction weeks: 104
Week 1/104 | Training rows: 1498 | Prediction rows: 12
Week 20/104 | Training rows: 1493 | Prediction rows: 15
Week 40/104 | Training rows: 1485 | Prediction rows: 15
Week 60/104 | Training rows: 1492 | Prediction rows: 15
Week 80/104 | Training rows: 1484 | Prediction rows: 15
Week 100/104 | Training rows: 1482 | Prediction rows: 15
Week 104/104 | Training rows: 1491 | Prediction rows: 3

=== ROLLING 104-WEEK VALIDATION RESULTS ===


,Country,Configuration,Strategy,Validation_Rows,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall,Baseline_Accuracy,Baseline_Macro_F1
0,Malaysia,V5_H5_M2,Weekly_Rolling_104W,4821,0.5306,0.4551,0.4584,0.2321,0.2767,0.7834,0.3151,0.4798,0.2161
1,Singapore,V5_H5_M2,Weekly_Rolling_104W,1488,0.4577,0.4168,0.4242,0.1573,0.3826,0.6505,0.2395,0.4422,0.2044



=== TRAINING STRATEGY COMPARISON ===


,Country,Strategy,Accuracy,Macro_F1,Balanced_Accuracy,MCC
0,Malaysia,Fixed,0.5401,0.4403,0.4674,0.2553
1,Malaysia,Weekly_Expanding,0.5414,0.4451,0.4669,0.2531
2,Malaysia,Weekly_Rolling_104W,0.5306,0.4551,0.4584,0.2321
3,Singapore,Fixed,0.4866,0.4440,0.4465,0.1945
4,Singapore,Weekly_Expanding,0.4819,0.4337,0.4382,0.1854
5,Singapore,Weekly_Rolling_104W,0.4577,0.4168,0.4242,0.1573



=== ROLLING TRAINING HISTORY SUMMARY ===


,Weeks,Min_Training_Rows,Max_Training_Rows,First_Training_Date,Latest_Training_Label
Country,,,,,
Malaysia,105,4682,4848,2021-01-04,2024-12-27
Singapore,104,1479,1500,2021-01-04,2024-12-20



Completed.


**Compare additional rolling training windows**

In [8]:
ROLLING_WINDOWS = [52, 78, 156]

additional_predictions = []
additional_results = []
additional_training_logs = []

for window_weeks in ROLLING_WINDOWS:

    print(f"\n=== ROLLING {window_weeks} WEEKS ===")

    window_prediction_records = []

    for country in ["Malaysia", "Singapore"]:

        country_history = history_data.loc[
            history_data["Country"] == country
        ]

        country_validation = validation_data.loc[
            validation_data["Country"] == country
        ]

        weekly_groups = list(
            country_validation.groupby("Week_Start", sort=True)
        )

        for week_number, (week_start, week_data) in enumerate(
            weekly_groups,
            start=1
        ):

            # Define rolling training boundaries
            rolling_start = (
                week_start - pd.Timedelta(weeks=window_weeks)
            )

            # Select completed events within the rolling window
            train = country_history.loc[
                (country_history["Date"] >= rolling_start)
                & (country_history["Date"] < week_start)
                & (country_history["Event_End_Date"] < week_start)
            ]

            assert len(train) > 0
            assert train["Event_End_Date"].max() < week_start

            X_train = train[feature_columns]
            y_train = train["Label"].to_numpy() + 1

            X_valid = week_data[feature_columns]

            assert set(np.unique(y_train)) == {0, 1, 2}

            # Train one model at the beginning of each week
            model = XGBClassifier(**model_params)

            model.fit(
                X_train,
                y_train,
                verbose=False
            )

            predictions = model.predict(X_valid) - 1

            majority_class = int(
                train["Label"].value_counts().idxmax()
            )

            # Store daily predictions
            week_predictions = week_data[
                [
                    "Ticker",
                    "Country",
                    "Date",
                    "Event_End_Date",
                    "Label",
                    "Week_Start"
                ]
            ].copy()

            week_predictions["Configuration"] = CONFIGURATION
            week_predictions["Predicted_Label"] = predictions
            week_predictions["Baseline_Label"] = majority_class
            week_predictions["Training_Strategy"] = (
                f"Rolling_{window_weeks}W"
            )

            window_prediction_records.append(week_predictions)

            additional_training_logs.append({
                "Country": country,
                "Window_Weeks": window_weeks,
                "Week_Start": week_start,
                "Training_Rows": len(train),
                "Prediction_Rows": len(week_data),
                "Latest_Training_Event_End": (
                    train["Event_End_Date"].max()
                )
            })

            if (
                week_number == 1
                or week_number % 25 == 0
                or week_number == len(weekly_groups)
            ):
                print(
                    f"{country} | "
                    f"Week {week_number}/{len(weekly_groups)} | "
                    f"Training rows: {len(train)}"
                )

    # Combine predictions for the current window
    window_predictions = pd.concat(
        window_prediction_records,
        ignore_index=True
    )

    assert len(window_predictions) == len(validation_data)

    # Verify identical validation targets across strategies
    window_keys = (
        window_predictions[["Ticker", "Date", "Label"]]
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )

    assert window_keys.equals(expanding_keys)

    additional_predictions.append(window_predictions)

    # Evaluate the current rolling window
    for country, subset in window_predictions.groupby("Country"):

        y_true = subset["Label"].to_numpy()
        y_pred = subset["Predicted_Label"].to_numpy()

        additional_results.append({
            "Country": country,
            "Configuration": CONFIGURATION,
            "Strategy": f"Rolling_{window_weeks}W",
            "Window_Weeks": window_weeks,
            "Validation_Rows": len(subset),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "MCC": matthews_corrcoef(y_true, y_pred),
            "Down_Recall": np.mean(
                y_pred[y_true == -1] == -1
            ),
            "Flat_Recall": np.mean(
                y_pred[y_true == 0] == 0
            ),
            "Up_Recall": np.mean(
                y_pred[y_true == 1] == 1
            )
        })

# Consolidate additional rolling-window results
additional_rolling_results = pd.DataFrame(
    additional_results
)

additional_rolling_predictions = pd.concat(
    additional_predictions,
    ignore_index=True
)

additional_training_log = pd.DataFrame(
    additional_training_logs
)

# Combine fixed, expanding and all rolling strategies
all_strategy_results = pd.concat(
    [
        strategy_comparison,
        additional_rolling_results[
            [
                "Country",
                "Strategy",
                "Accuracy",
                "Macro_F1",
                "Balanced_Accuracy",
                "MCC"
            ]
        ]
    ],
    ignore_index=True
)

all_strategy_results = (
    all_strategy_results
    .sort_values(
        ["Country", "Macro_F1"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\n=== ADDITIONAL ROLLING WINDOW RESULTS ===")
display(additional_rolling_results.round(4))

print("\n=== ALL TRAINING STRATEGIES ===")
display(all_strategy_results.round(4))

# Save experiment outputs
additional_rolling_results.to_csv(
    RESULTS_DIR / "triple_barrier_additional_rolling_results.csv",
    index=False
)

additional_rolling_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_additional_rolling_predictions.csv",
    index=False
)

additional_training_log.to_csv(
    RESULTS_DIR / "triple_barrier_additional_rolling_training_log.csv",
    index=False
)

all_strategy_results.to_csv(
    RESULTS_DIR / "triple_barrier_all_training_strategies.csv",
    index=False
)

print("\nCompleted.")


=== ROLLING 52 WEEKS ===
Malaysia | Week 1/105 | Training rows: 2391
Malaysia | Week 25/105 | Training rows: 2377
Malaysia | Week 50/105 | Training rows: 2388
Malaysia | Week 75/105 | Training rows: 2403
Malaysia | Week 100/105 | Training rows: 2375
Malaysia | Week 105/105 | Training rows: 2370
Singapore | Week 1/104 | Training rows: 739
Singapore | Week 25/104 | Training rows: 739
Singapore | Week 50/104 | Training rows: 738
Singapore | Week 75/104 | Training rows: 736
Singapore | Week 100/104 | Training rows: 735
Singapore | Week 104/104 | Training rows: 744

=== ROLLING 78 WEEKS ===
Malaysia | Week 1/105 | Training rows: 3574
Malaysia | Week 25/105 | Training rows: 3587
Malaysia | Week 50/105 | Training rows: 3614
Malaysia | Week 75/105 | Training rows: 3601
Malaysia | Week 100/105 | Training rows: 3613
Malaysia | Week 105/105 | Training rows: 3617
Singapore | Week 1/104 | Training rows: 1120
Singapore | Week 25/104 | Training rows: 1111
Singapore | Week 50/104 | Training rows: 111

,Country,Configuration,Strategy,Window_Weeks,Validation_Rows,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,V5_H5_M2,Rolling_52W,52,4821,0.5206,0.4468,0.4500,0.2162,0.2550,0.7644,0.3306
1,Singapore,V5_H5_M2,Rolling_52W,52,1488,0.4429,0.4117,0.4170,0.1438,0.4011,0.5973,0.2528
2,Malaysia,V5_H5_M2,Rolling_78W,78,4821,0.5173,0.4396,0.4439,0.2099,0.2489,0.7722,0.3107
3,Singapore,V5_H5_M2,Rolling_78W,78,1488,0.4355,0.4051,0.4110,0.1305,0.4011,0.5836,0.2483
4,Malaysia,V5_H5_M2,Rolling_156W,156,4821,0.5329,0.4596,0.4628,0.2381,0.3044,0.7821,0.3018
5,Singapore,V5_H5_M2,Rolling_156W,156,1488,0.4603,0.4149,0.4276,0.1616,0.4222,0.6611,0.1996



=== ALL TRAINING STRATEGIES ===


,Country,Strategy,Accuracy,Macro_F1,Balanced_Accuracy,MCC
0,Malaysia,Rolling_156W,0.5329,0.4596,0.4628,0.2381
1,Malaysia,Weekly_Rolling_104W,0.5306,0.4551,0.4584,0.2321
2,Malaysia,Rolling_52W,0.5206,0.4468,0.4500,0.2162
3,Malaysia,Weekly_Expanding,0.5414,0.4451,0.4669,0.2531
4,Malaysia,Fixed,0.5401,0.4403,0.4674,0.2553
5,Malaysia,Rolling_78W,0.5173,0.4396,0.4439,0.2099
6,Singapore,Fixed,0.4866,0.4440,0.4465,0.1945
7,Singapore,Weekly_Expanding,0.4819,0.4337,0.4382,0.1854
8,Singapore,Weekly_Rolling_104W,0.4577,0.4168,0.4242,0.1573
9,Singapore,Rolling_156W,0.4603,0.4149,0.4276,0.1616



Completed.


**Recalculate results using complete validation horizons**

In [9]:
HORIZON = 5
VALIDATION_END = pd.Timestamp("2024-12-31")

# Identify the complete future observation window
calendar = all_features.loc[
    all_features["Date"] <= VALIDATION_END,
    ["Ticker", "Segment_ID", "Date"]
].copy()

calendar = calendar.sort_values(
    ["Ticker", "Segment_ID", "Date"]
)

calendar["Full_Horizon_End"] = (
    calendar.groupby(["Ticker", "Segment_ID"])["Date"]
    .shift(-HORIZON)
)

# Collect predictions from all training strategies
prediction_sets = {
    "Fixed": validation_predictions.loc[
        validation_predictions["Configuration"] == "V5_H5_M2"
    ].copy(),
    "Weekly_Expanding": expanding_predictions.copy(),
    "Weekly_Rolling_104W": rolling_predictions.copy()
}

for window in [52, 78, 156]:
    strategy = f"Rolling_{window}W"

    prediction_sets[strategy] = (
        additional_rolling_predictions.loc[
            additional_rolling_predictions["Training_Strategy"]
            == strategy
        ].copy()
    )

corrected_records = []
corrected_results = []
reference_keys = None

for strategy, predictions in prediction_sets.items():

    # Match predictions to complete future horizons
    checked = predictions.merge(
        calendar[
            ["Ticker", "Date", "Full_Horizon_End"]
        ],
        on=["Ticker", "Date"],
        how="left",
        validate="many_to_one"
    )

    # Retain only events with complete validation horizons
    checked = checked.loc[
        checked["Full_Horizon_End"].notna()
    ].copy()

    assert (
        checked["Event_End_Date"]
        <= checked["Full_Horizon_End"]
    ).all()

    checked["Strategy"] = strategy

    # Verify identical observations across strategies
    keys = (
        checked[["Ticker", "Date", "Label"]]
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )

    if reference_keys is None:
        reference_keys = keys
    else:
        assert keys.equals(reference_keys)

    corrected_records.append(checked)

    for country, subset in checked.groupby("Country"):

        y_true = subset["Label"].to_numpy()
        y_pred = subset["Predicted_Label"].to_numpy()

        original_count = int(
            (predictions["Country"] == country).sum()
        )

        corrected_results.append({
            "Country": country,
            "Strategy": strategy,
            "Original_Rows": original_count,
            "Corrected_Rows": len(subset),
            "Excluded_Rows": original_count - len(subset),
            "Prediction_Weeks": (
                subset["Date"]
                .dt.to_period("W-SUN")
                .nunique()
            ),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "MCC": matthews_corrcoef(y_true, y_pred),
            "Down_Recall": np.mean(
                y_pred[y_true == -1] == -1
            ),
            "Flat_Recall": np.mean(
                y_pred[y_true == 0] == 0
            ),
            "Up_Recall": np.mean(
                y_pred[y_true == 1] == 1
            )
        })

# Assemble corrected results
corrected_predictions = pd.concat(
    corrected_records,
    ignore_index=True
)

corrected_results_df = pd.DataFrame(
    corrected_results
)

corrected_results_df = corrected_results_df.sort_values(
    ["Country", "Macro_F1"],
    ascending=[True, False]
).reset_index(drop=True)

print("=== CORRECTED VALIDATION SAMPLE ===")

display(
    corrected_results_df[
        [
            "Country",
            "Strategy",
            "Original_Rows",
            "Corrected_Rows",
            "Excluded_Rows",
            "Prediction_Weeks"
        ]
    ]
)

print("\n=== CORRECTED TRAINING STRATEGY COMPARISON ===")

display(
    corrected_results_df[
        [
            "Country",
            "Strategy",
            "Accuracy",
            "Macro_F1",
            "Balanced_Accuracy",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Save corrected evaluation
corrected_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_corrected_validation_predictions.csv",
    index=False
)

corrected_results_df.to_csv(
    RESULTS_DIR / "triple_barrier_corrected_strategy_comparison.csv",
    index=False
)

print("\nCompleted.")

=== CORRECTED VALIDATION SAMPLE ===


,Country,Strategy,Original_Rows,Corrected_Rows,Excluded_Rows,Prediction_Weeks
0,Malaysia,Rolling_156W,4821,4814,7,104
1,Malaysia,Weekly_Rolling_104W,4821,4814,7,104
2,Malaysia,Rolling_52W,4821,4814,7,104
3,Malaysia,Weekly_Expanding,4821,4814,7,104
4,Malaysia,Fixed,4821,4814,7,104
5,Malaysia,Rolling_78W,4821,4814,7,104
6,Singapore,Fixed,1488,1488,0,104
7,Singapore,Weekly_Expanding,1488,1488,0,104
8,Singapore,Weekly_Rolling_104W,1488,1488,0,104
9,Singapore,Rolling_156W,1488,1488,0,104



=== CORRECTED TRAINING STRATEGY COMPARISON ===


,Country,Strategy,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,Rolling_156W,0.5334,0.4600,0.4630,0.2387,0.3038,0.7821,0.3032
1,Malaysia,Weekly_Rolling_104W,0.5314,0.4557,0.4589,0.2330,0.2769,0.7834,0.3165
2,Malaysia,Rolling_52W,0.5214,0.4475,0.4506,0.2172,0.2552,0.7644,0.3321
3,Malaysia,Weekly_Expanding,0.5420,0.4454,0.4670,0.2535,0.4184,0.8314,0.1512
4,Malaysia,Fixed,0.5407,0.4407,0.4675,0.2556,0.4436,0.8284,0.1305
5,Malaysia,Rolling_78W,0.5181,0.4402,0.4445,0.2109,0.2491,0.7722,0.3121
6,Singapore,Fixed,0.4866,0.4440,0.4465,0.1945,0.3140,0.6884,0.3370
7,Singapore,Weekly_Expanding,0.4819,0.4337,0.4382,0.1854,0.3087,0.7067,0.2993
8,Singapore,Weekly_Rolling_104W,0.4577,0.4168,0.4242,0.1573,0.3826,0.6505,0.2395
9,Singapore,Rolling_156W,0.4603,0.4149,0.4276,0.1616,0.4222,0.6611,0.1996



Completed.


**Compare training strategies across years and banks**

In [10]:
# Define evaluation metrics
def summarise_performance(group):
    y_true = group["Label"].to_numpy()
    y_pred = group["Predicted_Label"].to_numpy()

    result = {
        "Observations": len(group),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

    for label, name in [
        (-1, "Down"),
        (0, "Flat"),
        (1, "Up")
    ]:
        mask = y_true == label

        result[f"{name}_Recall"] = (
            np.mean(y_pred[mask] == label)
            if mask.any()
            else np.nan
        )

    return result


# Calculate annual performance for every strategy
evaluation_data = corrected_predictions.copy()
evaluation_data["Year"] = evaluation_data["Date"].dt.year

yearly_records = []

for (country, strategy, year), group in evaluation_data.groupby(
    ["Country", "Strategy", "Year"]
):
    yearly_records.append({
        "Country": country,
        "Strategy": strategy,
        "Year": year,
        **summarise_performance(group)
    })

yearly_strategy_results = pd.DataFrame(yearly_records)

print("=== YEARLY STRATEGY PERFORMANCE ===")

display(
    yearly_strategy_results.sort_values(
        ["Country", "Year", "Macro_F1"],
        ascending=[True, True, False]
    ).round(4)
)

# Calculate bank-level performance
bank_records = []

for (country, strategy, ticker), group in evaluation_data.groupby(
    ["Country", "Strategy", "Ticker"]
):
    bank_records.append({
        "Country": country,
        "Strategy": strategy,
        "Ticker": ticker,
        **summarise_performance(group)
    })

bank_strategy_results = pd.DataFrame(bank_records)

# Compare selected candidate strategies
candidate_strategies = {
    "Malaysia": ["Fixed", "Rolling_104W", "Rolling_156W"],
    "Singapore": ["Fixed", "Weekly_Expanding", "Rolling_156W"]
}

candidate_rows = pd.concat([
    bank_strategy_results.loc[
        (bank_strategy_results["Country"] == country)
        & (
            bank_strategy_results["Strategy"].isin(strategies)
        )
    ]
    for country, strategies in candidate_strategies.items()
])

print("\n=== BANK-LEVEL CANDIDATE COMPARISON ===")

display(
    candidate_rows[
        [
            "Country",
            "Ticker",
            "Strategy",
            "Observations",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].sort_values(
        ["Country", "Ticker", "Strategy"]
    ).round(4)
)

# Identify the highest Macro F1 strategy for each year
yearly_winners = (
    yearly_strategy_results
    .sort_values(
        "Macro_F1",
        ascending=False
    )
    .groupby(["Country", "Year"])
    .head(1)
    .sort_values(["Country", "Year"])
)

print("\n=== HIGHEST MACRO F1 BY YEAR ===")

display(
    yearly_winners[
        [
            "Country",
            "Year",
            "Strategy",
            "Observations",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Save yearly and bank-level results
yearly_strategy_results.to_csv(
    RESULTS_DIR / "triple_barrier_strategy_yearly_comparison.csv",
    index=False
)

bank_strategy_results.to_csv(
    RESULTS_DIR / "triple_barrier_strategy_bank_comparison.csv",
    index=False
)

print("\nCompleted.")

=== YEARLY STRATEGY PERFORMANCE ===


,Country,Strategy,Year,Observations,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall
10,Malaysia,Weekly_Rolling_104W,2023,2435,0.5179,0.4564,0.4580,0.2238,0.2848,0.7288,0.3604
2,Malaysia,Rolling_156W,2023,2435,0.5175,0.4558,0.4572,0.2238,0.3312,0.7314,0.3089
8,Malaysia,Weekly_Expanding,2023,2435,0.5326,0.4372,0.4598,0.2407,0.4544,0.7956,0.1295
0,Malaysia,Fixed,2023,2435,0.5351,0.4371,0.4618,0.2462,0.4640,0.7998,0.1217
4,Malaysia,Rolling_52W,2023,2435,0.4949,0.4331,0.4348,0.1870,0.2880,0.7074,0.3089
6,Malaysia,Rolling_78W,2023,2435,0.4936,0.4289,0.4314,0.1844,0.2640,0.7134,0.3167
3,Malaysia,Rolling_156W,2024,2379,0.5498,0.4624,0.4678,0.2554,0.2713,0.8339,0.2980
5,Malaysia,Rolling_52W,2024,2379,0.5485,0.4592,0.4640,0.2495,0.2163,0.8226,0.3531
11,Malaysia,Weekly_Rolling_104W,2024,2379,0.5452,0.4544,0.4612,0.2456,0.2676,0.8392,0.2768
7,Malaysia,Rolling_78W,2024,2379,0.5431,0.4509,0.4572,0.2407,0.2315,0.8322,0.3079



=== BANK-LEVEL CANDIDATE COMPARISON ===


,Country,Ticker,Strategy,Observations,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,1015.KL,Fixed,486,0.3859,0.1991,0.3178,0.8707,0.0884
10,Malaysia,1015.KL,Rolling_156W,486,0.4383,0.2236,0.2523,0.8319,0.2585
1,Malaysia,1023.KL,Fixed,486,0.4528,0.2598,0.4390,0.8213,0.1795
11,Malaysia,1023.KL,Rolling_156W,486,0.4335,0.1993,0.3008,0.7536,0.2756
2,Malaysia,1066.KL,Fixed,486,0.4203,0.2754,0.5500,0.7891,0.0692
12,Malaysia,1066.KL,Rolling_156W,486,0.4902,0.2881,0.3900,0.7031,0.3846
3,Malaysia,1155.KL,Fixed,486,0.4538,0.2702,0.5500,0.7565,0.1397
13,Malaysia,1155.KL,Rolling_156W,486,0.4687,0.2409,0.4833,0.6435,0.2941
4,Malaysia,1171.KL,Fixed,483,0.4457,0.2641,0.2609,0.9064,0.2105
14,Malaysia,1171.KL,Rolling_156W,483,0.4613,0.2726,0.2174,0.8936,0.2932



=== HIGHEST MACRO F1 BY YEAR ===


,Country,Year,Strategy,Observations,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
10,Malaysia,2023,Weekly_Rolling_104W,2435,0.4564,0.2238,0.2848,0.7288,0.3604
3,Malaysia,2024,Rolling_156W,2379,0.4624,0.2554,0.2713,0.8339,0.2980
14,Singapore,2023,Rolling_156W,747,0.4151,0.1494,0.3636,0.6384,0.2525
13,Singapore,2024,Fixed,741,0.4817,0.2555,0.4257,0.7059,0.3399



Completed.


**Correct bank-level strategy comparison**

In [11]:
# Define candidate strategies using exact names
candidate_strategies = {
    "Malaysia": [
        "Fixed",
        "Weekly_Rolling_104W",
        "Rolling_156W"
    ],
    "Singapore": [
        "Fixed",
        "Weekly_Expanding",
        "Rolling_156W"
    ]
}

# Select candidate bank-level results
candidate_rows = pd.concat(
    [
        bank_strategy_results.loc[
            (bank_strategy_results["Country"] == country)
            & (
                bank_strategy_results["Strategy"]
                .isin(strategies)
            )
        ]
        for country, strategies in candidate_strategies.items()
    ],
    ignore_index=True
)

# Verify all candidate strategies are included
for country, strategies in candidate_strategies.items():
    actual = set(
        candidate_rows.loc[
            candidate_rows["Country"] == country,
            "Strategy"
        ]
    )

    assert actual == set(strategies)

assert len(candidate_rows) == 39

print("=== CORRECTED BANK-LEVEL COMPARISON ===")

display(
    candidate_rows[
        [
            "Country",
            "Ticker",
            "Strategy",
            "Observations",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].sort_values(
        ["Country", "Ticker", "Strategy"]
    ).round(4)
)

# Compare Malaysian rolling strategies against fixed training
malaysia_f1 = (
    candidate_rows.loc[
        candidate_rows["Country"] == "Malaysia"
    ]
    .pivot(
        index="Ticker",
        columns="Strategy",
        values="Macro_F1"
    )
)

malaysia_f1["156W_vs_Fixed"] = (
    malaysia_f1["Rolling_156W"]
    - malaysia_f1["Fixed"]
)

malaysia_f1["156W_vs_104W"] = (
    malaysia_f1["Rolling_156W"]
    - malaysia_f1["Weekly_Rolling_104W"]
)

print("\n=== MALAYSIA BANK-LEVEL MACRO F1 DIFFERENCES ===")

display(malaysia_f1.round(4))

print("\nBanks where 156W outperformed fixed:",
      int((malaysia_f1["156W_vs_Fixed"] > 0).sum()))

print("Banks where 156W outperformed 104W:",
      int((malaysia_f1["156W_vs_104W"] > 0).sum()))

# Save corrected comparison
candidate_rows.to_csv(
    RESULTS_DIR / "triple_barrier_candidate_bank_comparison.csv",
    index=False
)

malaysia_f1.to_csv(
    RESULTS_DIR / "triple_barrier_malaysia_rolling_comparison.csv"
)

print("\nCompleted.")

=== CORRECTED BANK-LEVEL COMPARISON ===


,Country,Ticker,Strategy,Observations,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,1015.KL,Fixed,486,0.3859,0.1991,0.3178,0.8707,0.0884
10,Malaysia,1015.KL,Rolling_156W,486,0.4383,0.2236,0.2523,0.8319,0.2585
20,Malaysia,1015.KL,Weekly_Rolling_104W,486,0.4457,0.2217,0.2897,0.8103,0.2585
1,Malaysia,1023.KL,Fixed,486,0.4528,0.2598,0.4390,0.8213,0.1795
11,Malaysia,1023.KL,Rolling_156W,486,0.4335,0.1993,0.3008,0.7536,0.2756
21,Malaysia,1023.KL,Weekly_Rolling_104W,486,0.4608,0.2434,0.2764,0.7923,0.3397
2,Malaysia,1066.KL,Fixed,486,0.4203,0.2754,0.5500,0.7891,0.0692
12,Malaysia,1066.KL,Rolling_156W,486,0.4902,0.2881,0.3900,0.7031,0.3846
22,Malaysia,1066.KL,Weekly_Rolling_104W,486,0.4493,0.2341,0.2700,0.7070,0.3692
3,Malaysia,1155.KL,Fixed,486,0.4538,0.2702,0.5500,0.7565,0.1397



=== MALAYSIA BANK-LEVEL MACRO F1 DIFFERENCES ===


Strategy,Fixed,Rolling_156W,Weekly_Rolling_104W,156W_vs_Fixed,156W_vs_104W
Ticker,,,,,
1015.KL,0.3859,0.4383,0.4457,0.0523,-0.0075
1023.KL,0.4528,0.4335,0.4608,-0.0193,-0.0273
1066.KL,0.4203,0.4902,0.4493,0.0699,0.0408
1155.KL,0.4538,0.4687,0.4767,0.0148,-0.0080
1171.KL,0.4457,0.4613,0.4428,0.0156,0.0186
1295.KL,0.4447,0.4776,0.4579,0.0329,0.0197
2488.KL,0.4525,0.4506,0.4333,-0.0019,0.0173
5185.KL,0.4441,0.4450,0.4477,0.0009,-0.0028
5258.KL,0.4098,0.4380,0.4449,0.0282,-0.0069



Banks where 156W outperformed fixed: 8
Banks where 156W outperformed 104W: 5

Completed.


**Audit the fixed-training boundary**

In [12]:
HORIZON = 5
TRAIN_END = pd.Timestamp("2022-12-31")

# Identify the complete five-day horizon for each observation
horizon_calendar = all_features[
    ["Ticker", "Segment_ID", "Date"]
].copy()

horizon_calendar = horizon_calendar.sort_values(
    ["Ticker", "Segment_ID", "Date"]
)

horizon_calendar["Full_Horizon_End"] = (
    horizon_calendar
    .groupby(["Ticker", "Segment_ID"])["Date"]
    .shift(-HORIZON)
)

# Match complete horizons to labelled historical observations
boundary_data = history_data.loc[
    history_data["Date"] <= TRAIN_END
].merge(
    horizon_calendar[
        ["Ticker", "Date", "Full_Horizon_End"]
    ],
    on=["Ticker", "Date"],
    how="left",
    validate="one_to_one"
)

# Compare event maturity with complete-horizon eligibility
boundary_data["Label_Matured"] = (
    boundary_data["Event_End_Date"] <= TRAIN_END
)

boundary_data["Full_Horizon_Eligible"] = (
    boundary_data["Full_Horizon_End"] <= TRAIN_END
)

boundary_data["Early_Exit_Only"] = (
    boundary_data["Label_Matured"]
    & ~boundary_data["Full_Horizon_Eligible"]
)

# Summarise training-boundary effects
boundary_summary = (
    boundary_data.groupby("Country")
    .agg(
        Available_Observations=("Date", "size"),
        Existing_Fixed_Training=("Label_Matured", "sum"),
        Complete_Horizon_Training=("Full_Horizon_Eligible", "sum"),
        Early_Exit_Only=("Early_Exit_Only", "sum")
    )
    .reset_index()
)

# Verify the original fixed-training counts
original_fixed_counts = (
    data.loc[
        (data["Configuration"] == "V5_H5_M2")
        & (data["Split"] == "Train_2015_2022")
    ]
    .groupby("Country")
    .size()
)

for _, row in boundary_summary.iterrows():
    assert (
        row["Existing_Fixed_Training"]
        == original_fixed_counts.loc[row["Country"]]
    )

assert (
    boundary_data.loc[
        boundary_data["Full_Horizon_Eligible"],
        "Label_Matured"
    ]
).all()

print("=== FIXED-TRAINING BOUNDARY AUDIT ===")
display(boundary_summary)

print("\n=== EARLY-EXIT EVENTS NEAR TRAINING BOUNDARY ===")

display(
    boundary_data.loc[
        boundary_data["Early_Exit_Only"],
        [
            "Country",
            "Ticker",
            "Date",
            "Event_End_Date",
            "Full_Horizon_End",
            "Label"
        ]
    ].sort_values(["Country", "Ticker", "Date"])
)

# Save boundary audit
boundary_summary.to_csv(
    RESULTS_DIR / "triple_barrier_training_boundary_audit.csv",
    index=False
)

print("\nCompleted.")

=== FIXED-TRAINING BOUNDARY AUDIT ===


,Country,Available_Observations,Existing_Fixed_Training,Complete_Horizon_Training,Early_Exit_Only
0,Malaysia,18554,18519,18504,15
1,Singapore,5919,5908,5904,4



=== EARLY-EXIT EVENTS NEAR TRAINING BOUNDARY ===


,Country,Ticker,Date,Event_End_Date,Full_Horizon_End,Label
18504,Malaysia,1015.KL,2022-12-23,2022-12-27,2023-01-03,-1
18505,Malaysia,1023.KL,2022-12-23,2022-12-27,2023-01-03,-1
18515,Malaysia,1023.KL,2022-12-27,2022-12-29,2023-01-04,1
18525,Malaysia,1023.KL,2022-12-28,2022-12-29,2023-01-05,1
18506,Malaysia,1066.KL,2022-12-23,2022-12-28,2023-01-03,-1
18516,Malaysia,1066.KL,2022-12-27,2022-12-30,2023-01-04,1
18526,Malaysia,1066.KL,2022-12-28,2022-12-29,2023-01-05,1
18507,Malaysia,1155.KL,2022-12-23,2022-12-27,2023-01-03,-1
18509,Malaysia,1295.KL,2022-12-23,2022-12-27,2023-01-03,-1
18520,Malaysia,2488.KL,2022-12-27,2022-12-30,2023-01-04,-1



Completed.


**Test sensitivity to complete-horizon training eligibility**

In [13]:
sensitivity_records = []
sensitivity_predictions = []

for country in ["Malaysia", "Singapore"]:

    # Original training data using known event outcomes
    original_train = boundary_data.loc[
        (boundary_data["Country"] == country)
        & boundary_data["Label_Matured"]
    ].copy()

    # Conservative training data using complete horizons
    conservative_train = boundary_data.loc[
        (boundary_data["Country"] == country)
        & boundary_data["Full_Horizon_Eligible"]
    ].copy()

    assert (
        conservative_train["Full_Horizon_End"] <= TRAIN_END
    ).all()

    # Retrieve the corrected fixed-model validation sample
    fixed_valid = corrected_predictions.loc[
        (corrected_predictions["Country"] == country)
        & (corrected_predictions["Strategy"] == "Fixed")
    ].copy()

    # Restore model features for the same validation observations
    fixed_valid = fixed_valid.merge(
        history_data[
            ["Ticker", "Date"] + feature_columns
        ],
        on=["Ticker", "Date"],
        how="left",
        validate="one_to_one"
    )

    assert fixed_valid[feature_columns].notna().all().all()

    # Train XGBoost using conservative historical eligibility
    model = XGBClassifier(**model_params)

    model.fit(
        conservative_train[feature_columns],
        conservative_train["Label"].to_numpy() + 1,
        verbose=False
    )

    predictions = (
        model.predict(fixed_valid[feature_columns]) - 1
    )

    y_true = fixed_valid["Label"].to_numpy()
    original_predictions = (
        fixed_valid["Predicted_Label"].to_numpy()
    )

    # Calculate performance on identical validation observations
    original_macro_f1 = f1_score(
        y_true,
        original_predictions,
        labels=[-1, 0, 1],
        average="macro",
        zero_division=0
    )

    revised_macro_f1 = f1_score(
        y_true,
        predictions,
        labels=[-1, 0, 1],
        average="macro",
        zero_division=0
    )

    original_mcc = matthews_corrcoef(
        y_true,
        original_predictions
    )

    revised_mcc = matthews_corrcoef(
        y_true,
        predictions
    )

    sensitivity_records.append({
        "Country": country,
        "Original_Train_Rows": len(original_train),
        "Conservative_Train_Rows": len(conservative_train),
        "Excluded_Train_Rows": (
            len(original_train) - len(conservative_train)
        ),
        "Validation_Rows": len(fixed_valid),
        "Changed_Predictions": int(
            np.sum(original_predictions != predictions)
        ),
        "Original_Macro_F1": original_macro_f1,
        "Conservative_Macro_F1": revised_macro_f1,
        "Macro_F1_Change": (
            revised_macro_f1 - original_macro_f1
        ),
        "Original_MCC": original_mcc,
        "Conservative_MCC": revised_mcc,
        "MCC_Change": revised_mcc - original_mcc
    })

    # Store revised predictions for reproducibility
    prediction_output = fixed_valid[
        [
            "Ticker",
            "Country",
            "Date",
            "Event_End_Date",
            "Label"
        ]
    ].copy()

    prediction_output["Original_Predicted_Label"] = (
        original_predictions
    )

    prediction_output["Conservative_Predicted_Label"] = (
        predictions
    )

    sensitivity_predictions.append(prediction_output)

    print(
        f"{country}: "
        f"{len(original_train)} original training rows, "
        f"{len(conservative_train)} conservative training rows"
    )

# Combine sensitivity results
boundary_sensitivity = pd.DataFrame(
    sensitivity_records
)

boundary_sensitivity_predictions = pd.concat(
    sensitivity_predictions,
    ignore_index=True
)

print("\n=== FIXED TRAINING BOUNDARY SENSITIVITY ===")

display(boundary_sensitivity.round(5))

print("\n=== PREDICTION CHANGE SUMMARY ===")

display(
    boundary_sensitivity[
        [
            "Country",
            "Validation_Rows",
            "Changed_Predictions",
            "Macro_F1_Change",
            "MCC_Change"
        ]
    ].round(5)
)

# Save sensitivity outputs
boundary_sensitivity.to_csv(
    RESULTS_DIR / "triple_barrier_fixed_boundary_sensitivity.csv",
    index=False
)

boundary_sensitivity_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_fixed_boundary_sensitivity_predictions.csv",
    index=False
)

print("\nCompleted.")

Malaysia: 18519 original training rows, 18504 conservative training rows
Singapore: 5908 original training rows, 5904 conservative training rows

=== FIXED TRAINING BOUNDARY SENSITIVITY ===


,Country,Original_Train_Rows,Conservative_Train_Rows,Excluded_Train_Rows,Validation_Rows,Changed_Predictions,Original_Macro_F1,Conservative_Macro_F1,Macro_F1_Change,Original_MCC,Conservative_MCC,MCC_Change
0,Malaysia,18519,18504,15,4814,211,0.44067,0.44423,0.00356,0.25563,0.25656,0.00093
1,Singapore,5908,5904,4,1488,131,0.44403,0.44123,-0.00281,0.19454,0.19200,-0.00254



=== PREDICTION CHANGE SUMMARY ===


,Country,Validation_Rows,Changed_Predictions,Macro_F1_Change,MCC_Change
0,Malaysia,4814,211,0.00356,0.00093
1,Singapore,1488,131,-0.00281,-0.00254



Completed.


**Evaluate all configurations using complete validation horizons**

In [14]:
configuration_horizons = {
    "V5_H5_M2": 5,
    "V5_H10_M2": 10,
    "V10_H5_M2": 5,
    "V10_H10_M2": 10
}

VALIDATION_END = pd.Timestamp("2024-12-31")

# Identify full five-day and ten-day event horizons
configuration_calendar = all_features[
    ["Ticker", "Segment_ID", "Date"]
].copy()

configuration_calendar = configuration_calendar.sort_values(
    ["Ticker", "Segment_ID", "Date"]
)

for horizon in [5, 10]:
    configuration_calendar[f"Horizon_End_{horizon}"] = (
        configuration_calendar
        .groupby(["Ticker", "Segment_ID"])["Date"]
        .shift(-horizon)
    )

# Evaluate the original fixed predictions on complete horizons
configuration_results = []
configuration_prediction_records = []

for configuration, horizon in configuration_horizons.items():

    predictions = validation_predictions.loc[
        validation_predictions["Configuration"] == configuration
    ].copy()

    predictions = predictions.merge(
        configuration_calendar[
            ["Ticker", "Date", f"Horizon_End_{horizon}"]
        ],
        on=["Ticker", "Date"],
        how="left",
        validate="many_to_one"
    )

    predictions = predictions.rename(
        columns={
            f"Horizon_End_{horizon}": "Full_Horizon_End"
        }
    )

    for country, original_group in predictions.groupby("Country"):

        # Retain predictions with a fully observable horizon
        valid = original_group.loc[
            original_group["Full_Horizon_End"].notna()
            & (
                original_group["Full_Horizon_End"]
                <= VALIDATION_END
            )
        ].copy()

        assert len(valid) > 0

        assert (
            valid["Event_End_Date"]
            <= valid["Full_Horizon_End"]
        ).all()

        y_true = valid["Label"].to_numpy()
        y_pred = valid["Predicted_Label"].to_numpy()

        result = {
            "Country": country,
            "Configuration": configuration,
            "Volatility_Window": int(
                configuration.split("_")[0][1:]
            ),
            "Holding_Days": horizon,
            "Original_Rows": len(original_group),
            "Corrected_Rows": len(valid),
            "Excluded_Rows": len(original_group) - len(valid),
            "Down_Share": np.mean(y_true == -1),
            "Flat_Share": np.mean(y_true == 0),
            "Up_Share": np.mean(y_true == 1),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "MCC": matthews_corrcoef(y_true, y_pred)
        }

        for label, name in [
            (-1, "Down"),
            (0, "Flat"),
            (1, "Up")
        ]:
            mask = y_true == label

            result[f"{name}_Recall"] = (
                np.mean(y_pred[mask] == label)
                if mask.any()
                else np.nan
            )

        configuration_results.append(result)

        valid["Training_Strategy"] = "Fixed"
        configuration_prediction_records.append(valid)

# Combine configuration results
fixed_configuration_results = pd.DataFrame(
    configuration_results
)

fixed_configuration_predictions = pd.concat(
    configuration_prediction_records,
    ignore_index=True
)

fixed_configuration_results = (
    fixed_configuration_results
    .sort_values(["Country", "Configuration"])
    .reset_index(drop=True)
)

print("=== VALIDATION HORIZON ELIGIBILITY ===")

display(
    fixed_configuration_results[
        [
            "Country",
            "Configuration",
            "Holding_Days",
            "Original_Rows",
            "Corrected_Rows",
            "Excluded_Rows"
        ]
    ]
)

print("\n=== TARGET CLASS DISTRIBUTIONS ===")

display(
    fixed_configuration_results[
        [
            "Country",
            "Configuration",
            "Down_Share",
            "Flat_Share",
            "Up_Share"
        ]
    ].round(4)
)

print("\n=== CORRECTED FIXED-MODEL PERFORMANCE ===")

display(
    fixed_configuration_results[
        [
            "Country",
            "Configuration",
            "Accuracy",
            "Macro_F1",
            "Balanced_Accuracy",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Confirm the previous V5_H5_M2 corrected results
for country in ["Malaysia", "Singapore"]:

    previous = corrected_results_df.loc[
        (corrected_results_df["Country"] == country)
        & (corrected_results_df["Strategy"] == "Fixed")
    ].iloc[0]

    current = fixed_configuration_results.loc[
        (fixed_configuration_results["Country"] == country)
        & (
            fixed_configuration_results["Configuration"]
            == "V5_H5_M2"
        )
    ].iloc[0]

    assert current["Corrected_Rows"] == previous["Corrected_Rows"]

    assert np.isclose(
        current["Macro_F1"],
        previous["Macro_F1"]
    )

# Save corrected fixed-configuration results
fixed_configuration_results.to_csv(
    RESULTS_DIR / "triple_barrier_fixed_configuration_corrected_results.csv",
    index=False
)

fixed_configuration_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_fixed_configuration_corrected_predictions.csv",
    index=False
)

print("\nCompleted.")

=== VALIDATION HORIZON ELIGIBILITY ===


,Country,Configuration,Holding_Days,Original_Rows,Corrected_Rows,Excluded_Rows
0,Malaysia,V10_H10_M2,10,4811,4765,46
1,Malaysia,V10_H5_M2,5,4829,4820,9
2,Malaysia,V5_H10_M2,10,4802,4759,43
3,Malaysia,V5_H5_M2,5,4821,4814,7
4,Singapore,V10_H10_M2,10,1481,1473,8
5,Singapore,V10_H5_M2,5,1488,1488,0
6,Singapore,V5_H10_M2,10,1479,1473,6
7,Singapore,V5_H5_M2,5,1488,1488,0



=== TARGET CLASS DISTRIBUTIONS ===


,Country,Configuration,Down_Share,Flat_Share,Up_Share
0,Malaysia,V10_H10_M2,0.3226,0.2837,0.3937
1,Malaysia,V10_H5_M2,0.2174,0.5301,0.2525
2,Malaysia,V5_H10_M2,0.3278,0.2629,0.4093
3,Malaysia,V5_H5_M2,0.2393,0.4805,0.2802
4,Singapore,V10_H10_M2,0.3293,0.2227,0.4481
5,Singapore,V10_H5_M2,0.2258,0.4812,0.2930
6,Singapore,V5_H10_M2,0.3530,0.1928,0.4542
7,Singapore,V5_H5_M2,0.2547,0.4422,0.3031



=== CORRECTED FIXED-MODEL PERFORMANCE ===


,Country,Configuration,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,V10_H10_M2,0.4063,0.4024,0.4133,0.1138,0.5999,0.3565,0.2836
1,Malaysia,V10_H5_M2,0.5432,0.3721,0.3989,0.1567,0.2204,0.8959,0.0805
2,Malaysia,V5_H10_M2,0.4383,0.4391,0.4557,0.1700,0.6404,0.4716,0.2551
3,Malaysia,V5_H5_M2,0.5407,0.4407,0.4675,0.2556,0.4436,0.8284,0.1305
4,Singapore,V10_H10_M2,0.4046,0.3608,0.3621,0.0339,0.4021,0.1524,0.5318
5,Singapore,V10_H5_M2,0.4724,0.3956,0.4007,0.1291,0.1815,0.7039,0.3165
6,Singapore,V5_H10_M2,0.4304,0.4062,0.3959,0.0705,0.4500,0.2430,0.4948
7,Singapore,V5_H5_M2,0.4866,0.4440,0.4465,0.1945,0.3140,0.6884,0.3370



Completed.


**Prepare chronological classification tuning datasets**

In [16]:
TUNE_CONFIGURATION = "V5_H5_M2"
TUNE_HORIZON = 5

INTERNAL_TRAIN_END = pd.Timestamp("2020-12-31")
INTERNAL_VALIDATION_START = pd.Timestamp("2021-01-01")
INTERNAL_VALIDATION_END = pd.Timestamp("2022-12-31")

FINAL_TRAIN_END = pd.Timestamp("2022-12-31")
EXTERNAL_VALIDATION_START = pd.Timestamp("2023-01-01")
EXTERNAL_VALIDATION_END = pd.Timestamp("2024-12-31")

# Reuse the labelled historical observations from Notebook 04
classification_tuning_data = history_data.loc[
    (history_data["Configuration"] == TUNE_CONFIGURATION)
    & (history_data["Date"] <= EXTERNAL_VALIDATION_END)
].copy()

# Calculate complete five-day horizons within the development period
tuning_calendar = all_features.loc[
    all_features["Date"] <= EXTERNAL_VALIDATION_END,
    ["Ticker", "Segment_ID", "Date"]
].copy()

tuning_calendar = tuning_calendar.sort_values(
    ["Ticker", "Segment_ID", "Date"]
)

tuning_calendar["Full_Horizon_End"] = (
    tuning_calendar
    .groupby(["Ticker", "Segment_ID"])["Date"]
    .shift(-TUNE_HORIZON)
)

classification_tuning_data = classification_tuning_data.merge(
    tuning_calendar[
        ["Ticker", "Date", "Full_Horizon_End"]
    ],
    on=["Ticker", "Date"],
    how="left",
    validate="one_to_one"
)

# Verify historical data integrity
assert not classification_tuning_data.duplicated(
    ["Ticker", "Date"]
).any()

assert classification_tuning_data[
    feature_columns
].notna().all().all()

assert set(
    classification_tuning_data["Label"].unique()
) == {-1, 0, 1}

assert (
    classification_tuning_data["Event_End_Date"]
    >= classification_tuning_data["Date"]
).all()

# Define chronological data partitions
internal_tuning_train = classification_tuning_data.loc[
    (classification_tuning_data["Date"] <= INTERNAL_TRAIN_END)
    & (
        classification_tuning_data["Event_End_Date"]
        <= INTERNAL_TRAIN_END
    )
].copy()

internal_tuning_valid = classification_tuning_data.loc[
    (
        classification_tuning_data["Date"]
        >= INTERNAL_VALIDATION_START
    )
    & (
        classification_tuning_data["Full_Horizon_End"]
        <= INTERNAL_VALIDATION_END
    )
].copy()

final_classification_train = classification_tuning_data.loc[
    (classification_tuning_data["Date"] <= FINAL_TRAIN_END)
    & (
        classification_tuning_data["Event_End_Date"]
        <= FINAL_TRAIN_END
    )
].copy()

final_classification_valid = classification_tuning_data.loc[
    (
        classification_tuning_data["Date"]
        >= EXTERNAL_VALIDATION_START
    )
    & (
        classification_tuning_data["Full_Horizon_End"]
        <= EXTERNAL_VALIDATION_END
    )
].copy()

# Verify chronological boundaries
assert (
    internal_tuning_train["Event_End_Date"].max()
    < internal_tuning_valid["Date"].min()
)

assert (
    internal_tuning_valid["Full_Horizon_End"]
    <= INTERNAL_VALIDATION_END
).all()

assert (
    final_classification_train["Event_End_Date"]
    <= FINAL_TRAIN_END
).all()

assert (
    final_classification_valid["Full_Horizon_End"]
    <= EXTERNAL_VALIDATION_END
).all()

# Confirm the external validation sample matches Cell 9
reference_validation = corrected_predictions.loc[
    corrected_predictions["Strategy"] == "Fixed",
    ["Ticker", "Date", "Label"]
].sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

prepared_validation = final_classification_valid[
    ["Ticker", "Date", "Label"]
].sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

assert prepared_validation.equals(reference_validation)

# Assign weekly retraining dates
for subset in [
    internal_tuning_valid,
    final_classification_valid
]:
    subset["Week_Start"] = (
        subset["Date"]
        - pd.to_timedelta(
            subset["Date"].dt.weekday,
            unit="D"
        )
    )

# Summarise the tuning and evaluation periods
periods = {
    "Internal_Train_2015_2020": internal_tuning_train,
    "Internal_Validation_2021_2022": internal_tuning_valid,
    "Final_Train_2015_2022": final_classification_train,
    "External_Validation_2023_2024": final_classification_valid
}

tuning_summary_records = []

for period_name, subset in periods.items():

    for country, group in subset.groupby("Country"):

        tuning_summary_records.append({
            "Country": country,
            "Period": period_name,
            "Observations": len(group),
            "Stocks": group["Ticker"].nunique(),
            "First_Date": group["Date"].min(),
            "Last_Date": group["Date"].max(),
            "Down_Share": np.mean(group["Label"] == -1),
            "Flat_Share": np.mean(group["Label"] == 0),
            "Up_Share": np.mean(group["Label"] == 1)
        })

classification_tuning_summary = pd.DataFrame(
    tuning_summary_records
)

print("=== CLASSIFICATION TUNING PERIODS ===")
print("Configuration:", TUNE_CONFIGURATION)
print("Technical features:", len(feature_columns))

display(
    classification_tuning_summary[
        [
            "Country",
            "Period",
            "Observations",
            "Stocks",
            "First_Date",
            "Last_Date"
        ]
    ]
)

print("\n=== CLASS DISTRIBUTIONS BY PERIOD ===")

display(
    classification_tuning_summary[
        [
            "Country",
            "Period",
            "Down_Share",
            "Flat_Share",
            "Up_Share"
        ]
    ].round(4)
)

print("\n=== INTERNAL VALIDATION WEEKS ===")

display(
    internal_tuning_valid.groupby("Country").agg(
        Prediction_Weeks=("Week_Start", "nunique"),
        Prediction_Rows=("Date", "size"),
        First_Prediction=("Date", "min"),
        Last_Prediction=("Date", "max")
    )
)

# Save the data-partition summary
classification_tuning_summary.to_csv(
    RESULTS_DIR / "triple_barrier_classification_tuning_periods.csv",
    index=False
)

print("\nCompleted.")

=== CLASSIFICATION TUNING PERIODS ===
Configuration: V5_H5_M2
Technical features: 23


,Country,Period,Observations,Stocks,First_Date,Last_Date
0,Malaysia,Internal_Train_2015_2020,13763,10,2015-02-24,2020-12-30
1,Singapore,Internal_Train_2015_2020,4402,3,2015-02-18,2020-12-30
2,Malaysia,Internal_Validation_2021_2022,4699,10,2021-01-04,2022-12-22
3,Singapore,Internal_Validation_2021_2022,1494,3,2021-01-04,2022-12-22
4,Malaysia,Final_Train_2015_2022,18519,10,2015-02-24,2022-12-29
5,Singapore,Final_Train_2015_2022,5908,3,2015-02-18,2022-12-28
6,Malaysia,External_Validation_2023_2024,4814,10,2023-01-03,2024-12-23
7,Singapore,External_Validation_2023_2024,1488,3,2023-01-03,2024-12-23



=== CLASS DISTRIBUTIONS BY PERIOD ===


,Country,Period,Down_Share,Flat_Share,Up_Share
0,Malaysia,Internal_Train_2015_2020,0.2815,0.4624,0.2561
1,Singapore,Internal_Train_2015_2020,0.2865,0.4073,0.3062
2,Malaysia,Internal_Validation_2021_2022,0.2490,0.4769,0.2741
3,Singapore,Internal_Validation_2021_2022,0.2577,0.4592,0.2831
4,Malaysia,Final_Train_2015_2022,0.2740,0.4657,0.2603
5,Singapore,Final_Train_2015_2022,0.2793,0.4203,0.3004
6,Malaysia,External_Validation_2023_2024,0.2393,0.4805,0.2802
7,Singapore,External_Validation_2023_2024,0.2547,0.4422,0.3031



=== INTERNAL VALIDATION WEEKS ===


,Prediction_Weeks,Prediction_Rows,First_Prediction,Last_Prediction
Country,,,,
Malaysia,103,4699,2021-01-04,2022-12-22
Singapore,103,1494,2021-01-04,2022-12-22



Completed.


**Tune XGBoost classification using chronological validation**

In [17]:
classification_candidates = [
    {
        "Name": "Original",
        "Overrides": {},
        "Balanced_Weights": False
    },
    {
        "Name": "Depth2_Regularised",
        "Overrides": {
            "n_estimators": 150,
            "learning_rate": 0.03,
            "max_depth": 2,
            "min_child_weight": 10,
            "reg_lambda": 20,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        },
        "Balanced_Weights": False
    },
    {
        "Name": "Depth3_Regularised",
        "Overrides": {
            "n_estimators": 180,
            "learning_rate": 0.03,
            "max_depth": 3,
            "min_child_weight": 10,
            "reg_lambda": 30,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },
        "Balanced_Weights": False
    },
    {
        "Name": "Weighted_Depth2",
        "Overrides": {
            "n_estimators": 150,
            "learning_rate": 0.03,
            "max_depth": 2,
            "min_child_weight": 10,
            "reg_lambda": 20,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        },
        "Balanced_Weights": True
    }
]

classification_tuning_predictions = []
classification_tuning_results = []

ROLLING_WEEKS = 156

for country in ["Malaysia", "Singapore"]:

    country_history = classification_tuning_data.loc[
        classification_tuning_data["Country"] == country
    ].copy()

    country_valid = internal_tuning_valid.loc[
        internal_tuning_valid["Country"] == country
    ].copy()

    country_fixed_train = internal_tuning_train.loc[
        internal_tuning_train["Country"] == country
    ].copy()

    # Define prediction groups for each training strategy
    if country == "Malaysia":

        prediction_groups = list(
            country_valid.groupby("Week_Start", sort=True)
        )

        strategy = "Weekly_Rolling_156W"

    else:

        prediction_groups = [
            (
                INTERNAL_VALIDATION_START,
                country_valid
            )
        ]

        strategy = "Fixed"

    print(f"\n=== {country.upper()} ===")
    print("Training strategy:", strategy)
    print("Prediction groups:", len(prediction_groups))

    for candidate in classification_candidates:

        candidate_name = candidate["Name"]

        parameters = {
            **model_params,
            **candidate["Overrides"]
        }

        candidate_prediction_records = []

        for group_number, (week_start, week_data) in enumerate(
            prediction_groups,
            start=1
        ):

            # Select historical training observations
            if country == "Malaysia":

                rolling_start = (
                    week_start
                    - pd.Timedelta(weeks=ROLLING_WEEKS)
                )

                train = country_history.loc[
                    (country_history["Date"] >= rolling_start)
                    & (country_history["Date"] < week_start)
                    & (
                        country_history["Event_End_Date"]
                        < week_start
                    )
                ].copy()

            else:

                train = country_fixed_train

            assert len(train) > 0

            assert (
                train["Event_End_Date"].max()
                < week_data["Date"].min()
            )

            X_train = train[feature_columns]
            y_train = train["Label"].to_numpy() + 1

            X_valid = week_data[feature_columns]

            assert set(np.unique(y_train)) == {0, 1, 2}

            # Calculate optional inverse-frequency class weights
            sample_weights = None

            if candidate["Balanced_Weights"]:

                class_counts = np.bincount(
                    y_train,
                    minlength=3
                )

                assert (class_counts > 0).all()

                sample_weights = (
                    len(y_train)
                    / (3.0 * class_counts[y_train])
                )

            # Train XGBoost with the selected configuration
            model = XGBClassifier(**parameters)

            model.fit(
                X_train,
                y_train,
                sample_weight=sample_weights,
                verbose=False
            )

            predictions = (
                model.predict(X_valid) - 1
            )

            majority_class = int(
                train["Label"].value_counts().idxmax()
            )

            # Store the predictions for the current group
            prediction_output = week_data[
                [
                    "Ticker",
                    "Country",
                    "Date",
                    "Label",
                    "Week_Start"
                ]
            ].copy()

            prediction_output["Candidate"] = candidate_name
            prediction_output["Strategy"] = strategy
            prediction_output["Predicted_Label"] = predictions
            prediction_output["Baseline_Label"] = majority_class
            prediction_output["Training_Rows"] = len(train)

            candidate_prediction_records.append(
                prediction_output
            )

            if (
                country == "Malaysia"
                and (
                    group_number == 1
                    or group_number % 25 == 0
                    or group_number == len(prediction_groups)
                )
            ):
                print(
                    f"{candidate_name} | "
                    f"Week {group_number}/{len(prediction_groups)} | "
                    f"Training rows: {len(train)}"
                )

        # Combine the candidate's internal validation predictions
        candidate_predictions = pd.concat(
            candidate_prediction_records,
            ignore_index=True
        )

        assert len(candidate_predictions) == len(country_valid)

        assert not candidate_predictions.duplicated(
            ["Ticker", "Date"]
        ).any()

        y_true = candidate_predictions["Label"].to_numpy()
        y_pred = (
            candidate_predictions["Predicted_Label"].to_numpy()
        )
        y_baseline = (
            candidate_predictions["Baseline_Label"].to_numpy()
        )

        # Evaluate the full internal validation period
        classification_tuning_results.append({
            "Country": country,
            "Candidate": candidate_name,
            "Strategy": strategy,
            "Validation_Rows": len(candidate_predictions),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro_F1": f1_score(
                y_true,
                y_pred,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_true,
                y_pred
            ),
            "MCC": matthews_corrcoef(y_true, y_pred),
            "Down_Recall": np.mean(
                y_pred[y_true == -1] == -1
            ),
            "Flat_Recall": np.mean(
                y_pred[y_true == 0] == 0
            ),
            "Up_Recall": np.mean(
                y_pred[y_true == 1] == 1
            ),
            "Baseline_Macro_F1": f1_score(
                y_true,
                y_baseline,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            )
        })

        classification_tuning_predictions.append(
            candidate_predictions
        )

        print(
            f"{candidate_name} | "
            f"Macro F1: "
            f"{classification_tuning_results[-1]['Macro_F1']:.4f}"
        )

# Combine tuning results
classification_tuning_results_df = pd.DataFrame(
    classification_tuning_results
)

classification_tuning_predictions_df = pd.concat(
    classification_tuning_predictions,
    ignore_index=True
)

# Rank candidates by internal validation Macro F1
classification_tuning_results_df = (
    classification_tuning_results_df
    .sort_values(
        ["Country", "Macro_F1", "MCC"],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

print("\n=== INTERNAL CLASSIFICATION TUNING RESULTS ===")

display(
    classification_tuning_results_df.round(4)
)

# Identify the highest-Macro-F1 candidate for each market
selected_classification_candidates = (
    classification_tuning_results_df
    .groupby("Country", sort=False)
    .head(1)
    .reset_index(drop=True)
)

print("\n=== SELECTED CANDIDATES ===")

display(
    selected_classification_candidates[
        [
            "Country",
            "Candidate",
            "Strategy",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Confirm equal prediction coverage across candidates
for country in ["Malaysia", "Singapore"]:

    subset = classification_tuning_predictions_df.loc[
        classification_tuning_predictions_df["Country"] == country
    ]

    expected_rows = len(
        internal_tuning_valid.loc[
            internal_tuning_valid["Country"] == country
        ]
    ) * len(classification_candidates)

    assert len(subset) == expected_rows

# Save tuning results
classification_tuning_results_df.to_csv(
    RESULTS_DIR / "triple_barrier_classification_tuning_results.csv",
    index=False
)

classification_tuning_predictions_df.to_csv(
    RESULTS_DIR / "triple_barrier_classification_tuning_predictions.csv",
    index=False
)

selected_classification_candidates.to_csv(
    RESULTS_DIR / "triple_barrier_classification_selected_candidates.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Training strategy: Weekly_Rolling_156W
Prediction groups: 103
Original | Week 1/103 | Training rows: 7177
Original | Week 25/103 | Training rows: 7185
Original | Week 50/103 | Training rows: 7152
Original | Week 75/103 | Training rows: 7185
Original | Week 100/103 | Training rows: 7191
Original | Week 103/103 | Training rows: 7152
Original | Macro F1: 0.4329
Depth2_Regularised | Week 1/103 | Training rows: 7177
Depth2_Regularised | Week 25/103 | Training rows: 7185
Depth2_Regularised | Week 50/103 | Training rows: 7152
Depth2_Regularised | Week 75/103 | Training rows: 7185
Depth2_Regularised | Week 100/103 | Training rows: 7191
Depth2_Regularised | Week 103/103 | Training rows: 7152
Depth2_Regularised | Macro F1: 0.4121
Depth3_Regularised | Week 1/103 | Training rows: 7177
Depth3_Regularised | Week 25/103 | Training rows: 7185
Depth3_Regularised | Week 50/103 | Training rows: 7152
Depth3_Regularised | Week 75/103 | Training rows: 7185
Depth3_Regularised | Week 100/103

,Country,Candidate,Strategy,Validation_Rows,Accuracy,Macro_F1,Balanced_Accuracy,MCC,Down_Recall,Flat_Recall,Up_Recall,Baseline_Macro_F1
0,Malaysia,Weighted_Depth2,Weekly_Rolling_156W,4699,0.5012,0.4590,0.4654,0.2286,0.4829,0.6439,0.2694,0.2153
1,Malaysia,Original,Weekly_Rolling_156W,4699,0.5280,0.4329,0.4514,0.2294,0.3795,0.8202,0.1545,0.2153
2,Malaysia,Depth3_Regularised,Weekly_Rolling_156W,4699,0.5308,0.4264,0.4536,0.2353,0.4171,0.8295,0.1141,0.2153
3,Malaysia,Depth2_Regularised,Weekly_Rolling_156W,4699,0.5322,0.4121,0.4493,0.2347,0.4154,0.8541,0.0784,0.2153
4,Singapore,Depth3_Regularised,Fixed,1494,0.5067,0.4454,0.4492,0.2127,0.2571,0.7478,0.3428,0.2098
5,Singapore,Original,Fixed,1494,0.4913,0.4374,0.4399,0.1946,0.2675,0.7070,0.3452,0.2098
6,Singapore,Weighted_Depth2,Fixed,1494,0.4645,0.4308,0.4293,0.1727,0.3299,0.6152,0.3428,0.2098
7,Singapore,Depth2_Regularised,Fixed,1494,0.4967,0.4232,0.4337,0.1939,0.1818,0.7551,0.3641,0.2098



=== SELECTED CANDIDATES ===


,Country,Candidate,Strategy,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,Weighted_Depth2,Weekly_Rolling_156W,0.4590,0.2286,0.4829,0.6439,0.2694
1,Singapore,Depth3_Regularised,Fixed,0.4454,0.2127,0.2571,0.7478,0.3428



Completed.


**Evaluate selected XGBoost configurations on 2023–2024**

In [18]:
tuned_prediction_records = []
tuned_training_records = []
tuned_result_records = []

for country in ["Malaysia", "Singapore"]:

    # Retrieve the candidate selected using internal validation
    selected_name = selected_classification_candidates.loc[
        selected_classification_candidates["Country"] == country,
        "Candidate"
    ].iloc[0]

    candidate = next(
        item for item in classification_candidates
        if item["Name"] == selected_name
    )

    parameters = {
        **model_params,
        **candidate["Overrides"]
    }

    country_history = classification_tuning_data.loc[
        classification_tuning_data["Country"] == country
    ].copy()

    country_valid = final_classification_valid.loc[
        final_classification_valid["Country"] == country
    ].copy()

    if country == "Malaysia":

        strategy = "Weekly_Rolling_156W"

        prediction_groups = list(
            country_valid.groupby("Week_Start", sort=True)
        )

    else:

        strategy = "Fixed"

        prediction_groups = [
            (EXTERNAL_VALIDATION_START, country_valid)
        ]

    print(f"\n=== {country.upper()} ===")
    print("Selected candidate:", selected_name)
    print("Training strategy:", strategy)
    print("Prediction groups:", len(prediction_groups))

    for group_number, (week_start, week_data) in enumerate(
        prediction_groups,
        start=1
    ):

        # Select the training history
        if country == "Malaysia":

            rolling_start = (
                week_start - pd.Timedelta(weeks=156)
            )

            train = country_history.loc[
                (country_history["Date"] >= rolling_start)
                & (country_history["Date"] < week_start)
                & (country_history["Event_End_Date"] < week_start)
            ].copy()

        else:

            train = final_classification_train.loc[
                final_classification_train["Country"] == country
            ].copy()

        # Check label maturity before model fitting
        assert len(train) > 0
        assert train["Event_End_Date"].max() < week_start

        X_train = train[feature_columns]
        y_train = train["Label"].to_numpy() + 1

        X_valid = week_data[feature_columns]

        assert set(np.unique(y_train)) == {0, 1, 2}
        assert X_train.notna().all().all()
        assert X_valid.notna().all().all()

        # Calculate class weights when required
        sample_weights = None

        if candidate["Balanced_Weights"]:

            class_counts = np.bincount(
                y_train,
                minlength=3
            )

            assert (class_counts > 0).all()

            sample_weights = (
                len(y_train)
                / (3.0 * class_counts[y_train])
            )

        # Train the selected XGBoost model
        model = XGBClassifier(**parameters)

        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weights,
            verbose=False
        )

        assert np.array_equal(
            model.classes_,
            np.array([0, 1, 2])
        )

        # Generate daily classification predictions
        probabilities = model.predict_proba(X_valid)
        predictions = np.argmax(probabilities, axis=1) - 1

        assert np.allclose(
            probabilities.sum(axis=1),
            1.0
        )

        # Store predictions and class probabilities
        prediction_output = week_data[
            [
                "Ticker",
                "Country",
                "Date",
                "Event_End_Date",
                "Label",
                "Week_Start"
            ]
        ].copy()

        prediction_output["Candidate"] = selected_name
        prediction_output["Strategy"] = strategy
        prediction_output["Predicted_Label"] = predictions

        prediction_output["Probability_Down"] = (
            probabilities[:, 0]
        )

        prediction_output["Probability_Flat"] = (
            probabilities[:, 1]
        )

        prediction_output["Probability_Up"] = (
            probabilities[:, 2]
        )

        tuned_prediction_records.append(prediction_output)

        # Record the historical training boundaries
        tuned_training_records.append({
            "Country": country,
            "Candidate": selected_name,
            "Strategy": strategy,
            "Week_Start": week_start,
            "Training_Rows": len(train),
            "Prediction_Rows": len(week_data),
            "Earliest_Training_Date": train["Date"].min(),
            "Latest_Training_Event_End": (
                train["Event_End_Date"].max()
            )
        })

        if (
            group_number == 1
            or group_number % 25 == 0
            or group_number == len(prediction_groups)
        ):
            print(
                f"Group {group_number}/{len(prediction_groups)} | "
                f"Training rows: {len(train)} | "
                f"Prediction rows: {len(week_data)}"
            )

# Combine predictions and training logs
tuned_classification_predictions = pd.concat(
    tuned_prediction_records,
    ignore_index=True
)

tuned_classification_training_log = pd.DataFrame(
    tuned_training_records
)

assert not tuned_classification_predictions.duplicated(
    ["Ticker", "Date"]
).any()

assert (
    tuned_classification_training_log["Latest_Training_Event_End"]
    < tuned_classification_training_log["Week_Start"]
).all()

# Evaluate each market against the existing untuned strategy
for country in ["Malaysia", "Singapore"]:

    subset = tuned_classification_predictions.loc[
        tuned_classification_predictions["Country"] == country
    ].copy()

    original_strategy = (
        "Rolling_156W"
        if country == "Malaysia"
        else "Fixed"
    )

    # Verify identical validation observations
    reference = corrected_predictions.loc[
        (corrected_predictions["Country"] == country)
        & (corrected_predictions["Strategy"] == original_strategy),
        ["Ticker", "Date", "Label"]
    ].sort_values(
        ["Ticker", "Date"]
    ).reset_index(drop=True)

    prepared = subset[
        ["Ticker", "Date", "Label"]
    ].sort_values(
        ["Ticker", "Date"]
    ).reset_index(drop=True)

    assert prepared.equals(reference)

    y_true = subset["Label"].to_numpy()
    y_pred = subset["Predicted_Label"].to_numpy()

    original_result = corrected_results_df.loc[
        (corrected_results_df["Country"] == country)
        & (
            corrected_results_df["Strategy"]
            == original_strategy
        )
    ].iloc[0]

    tuned_macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=[-1, 0, 1],
        average="macro",
        zero_division=0
    )

    tuned_mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    tuned_result_records.append({
        "Country": country,
        "Candidate": subset["Candidate"].iloc[0],
        "Strategy": subset["Strategy"].iloc[0],
        "Validation_Rows": len(subset),
        "Original_Macro_F1": original_result["Macro_F1"],
        "Tuned_Macro_F1": tuned_macro_f1,
        "Macro_F1_Change": (
            tuned_macro_f1 - original_result["Macro_F1"]
        ),
        "Original_MCC": original_result["MCC"],
        "Tuned_MCC": tuned_mcc,
        "MCC_Change": tuned_mcc - original_result["MCC"],
        "Tuned_Accuracy": accuracy_score(y_true, y_pred),
        "Tuned_Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "Down_Recall": np.mean(
            y_pred[y_true == -1] == -1
        ),
        "Flat_Recall": np.mean(
            y_pred[y_true == 0] == 0
        ),
        "Up_Recall": np.mean(
            y_pred[y_true == 1] == 1
        )
    })

tuned_classification_results_df = pd.DataFrame(
    tuned_result_records
)

print("\n=== TUNED CLASSIFICATION VALIDATION RESULTS ===")

display(
    tuned_classification_results_df.round(4)
)

print("\n=== TUNED PREDICTION DISTRIBUTIONS ===")

prediction_distribution = pd.crosstab(
    tuned_classification_predictions["Country"],
    tuned_classification_predictions["Predicted_Label"],
    normalize="index"
)

display(
    prediction_distribution.reindex(
        columns=[-1, 0, 1],
        fill_value=0
    ).round(4)
)

print("\n=== TRAINING HISTORY SUMMARY ===")

display(
    tuned_classification_training_log.groupby("Country").agg(
        Training_Groups=("Week_Start", "nunique"),
        Min_Training_Rows=("Training_Rows", "min"),
        Max_Training_Rows=("Training_Rows", "max"),
        Total_Predictions=("Prediction_Rows", "sum")
    )
)

# Save tuned classification results
tuned_classification_predictions.to_csv(
    RESULTS_DIR / "triple_barrier_tuned_validation_predictions.csv",
    index=False
)

tuned_classification_results_df.to_csv(
    RESULTS_DIR / "triple_barrier_tuned_validation_results.csv",
    index=False
)

tuned_classification_training_log.to_csv(
    RESULTS_DIR / "triple_barrier_tuned_training_log.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Selected candidate: Weighted_Depth2
Training strategy: Weekly_Rolling_156W
Prediction groups: 104
Group 1/104 | Training rows: 7174 | Prediction rows: 40
Group 25/104 | Training rows: 7140 | Prediction rows: 50
Group 50/104 | Training rows: 7127 | Prediction rows: 50
Group 75/104 | Training rows: 7209 | Prediction rows: 40
Group 100/104 | Training rows: 7237 | Prediction rows: 47
Group 104/104 | Training rows: 7253 | Prediction rows: 10

=== SINGAPORE ===
Selected candidate: Depth3_Regularised
Training strategy: Fixed
Prediction groups: 1
Group 1/1 | Training rows: 5908 | Prediction rows: 1488

=== TUNED CLASSIFICATION VALIDATION RESULTS ===


,Country,Candidate,Strategy,Validation_Rows,Original_Macro_F1,Tuned_Macro_F1,Macro_F1_Change,Original_MCC,Tuned_MCC,MCC_Change,Tuned_Accuracy,Tuned_Balanced_Accuracy,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,Weighted_Depth2,Weekly_Rolling_156W,4814,0.460,0.4473,-0.0126,0.2387,0.2199,-0.0188,0.4807,0.4645,0.5773,0.5685,0.2476
1,Singapore,Depth3_Regularised,Fixed,1488,0.444,0.4487,0.0046,0.1945,0.2085,0.0139,0.4973,0.4535,0.3325,0.7264,0.3016



=== TUNED PREDICTION DISTRIBUTIONS ===


Predicted_Label,-1,0,1
Country,,,
Malaysia,0.4161,0.3880,0.1959
Singapore,0.2379,0.5457,0.2164



=== TRAINING HISTORY SUMMARY ===


,Training_Groups,Min_Training_Rows,Max_Training_Rows,Total_Predictions
Country,,,,
Malaysia,104,7112,7253,4814
Singapore,1,5908,5908,1488



Completed.


**Compare tuned and original models by year and bank**

In [19]:
comparison_records = []

for country in ["Malaysia", "Singapore"]:

    original_strategy = (
        "Rolling_156W" if country == "Malaysia" else "Fixed"
    )

    original = corrected_predictions.loc[
        (corrected_predictions["Country"] == country)
        & (corrected_predictions["Strategy"] == original_strategy),
        ["Ticker", "Country", "Date", "Label", "Predicted_Label"]
    ].copy()

    original = original.rename(columns={
        "Label": "Original_Label",
        "Predicted_Label": "Original_Prediction"
    })

    tuned = tuned_classification_predictions.loc[
        tuned_classification_predictions["Country"] == country,
        ["Ticker", "Country", "Date", "Label", "Predicted_Label"]
    ].copy()

    tuned = tuned.rename(columns={
        "Predicted_Label": "Tuned_Prediction"
    })

    # Align predictions on identical stock-date observations
    matched = tuned.merge(
        original,
        on=["Ticker", "Country", "Date"],
        how="outer",
        validate="one_to_one",
        indicator=True
    )

    assert matched["_merge"].eq("both").all()
    assert matched["Label"].eq(matched["Original_Label"]).all()

    matched = matched.drop(
        columns=["_merge", "Original_Label"]
    )

    matched["Year"] = matched["Date"].dt.year

    matched["Prediction_Changed"] = (
        matched["Original_Prediction"]
        != matched["Tuned_Prediction"]
    )

    matched["Original_Correct"] = (
        matched["Original_Prediction"] == matched["Label"]
    )

    matched["Tuned_Correct"] = (
        matched["Tuned_Prediction"] == matched["Label"]
    )

    comparison_records.append(matched)

classification_model_comparison = pd.concat(
    comparison_records,
    ignore_index=True
)

# Calculate metrics on identical observations
def calculate_classification_metrics(group, prediction_column):

    actual = group["Label"].to_numpy()
    predicted = group[prediction_column].to_numpy()

    result = {
        "Accuracy": accuracy_score(actual, predicted),
        "Macro_F1": f1_score(
            actual,
            predicted,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(actual, predicted)
    }

    for label, name in [
        (-1, "Down"),
        (0, "Flat"),
        (1, "Up")
    ]:
        mask = actual == label

        result[f"{name}_Recall"] = (
            np.mean(predicted[mask] == label)
            if mask.any()
            else np.nan
        )

    return result


# Compare yearly performance
yearly_records = []

for (country, year), group in classification_model_comparison.groupby(
    ["Country", "Year"]
):

    for method, prediction_column in [
        ("Original", "Original_Prediction"),
        ("Tuned", "Tuned_Prediction")
    ]:

        yearly_records.append({
            "Country": country,
            "Year": year,
            "Method": method,
            "Observations": len(group),
            **calculate_classification_metrics(
                group,
                prediction_column
            )
        })

tuned_yearly_comparison = pd.DataFrame(yearly_records)

print("=== ORIGINAL VS TUNED PERFORMANCE BY YEAR ===")

display(
    tuned_yearly_comparison.sort_values(
        ["Country", "Year", "Method"]
    ).round(4)
)


# Compare bank-level Macro F1
bank_records = []

for (country, ticker), group in classification_model_comparison.groupby(
    ["Country", "Ticker"]
):

    original_metrics = calculate_classification_metrics(
        group,
        "Original_Prediction"
    )

    tuned_metrics = calculate_classification_metrics(
        group,
        "Tuned_Prediction"
    )

    bank_records.append({
        "Country": country,
        "Ticker": ticker,
        "Observations": len(group),
        "Original_Macro_F1": original_metrics["Macro_F1"],
        "Tuned_Macro_F1": tuned_metrics["Macro_F1"],
        "Macro_F1_Change": (
            tuned_metrics["Macro_F1"]
            - original_metrics["Macro_F1"]
        ),
        "Original_MCC": original_metrics["MCC"],
        "Tuned_MCC": tuned_metrics["MCC"],
        "MCC_Change": (
            tuned_metrics["MCC"]
            - original_metrics["MCC"]
        )
    })

tuned_bank_comparison = pd.DataFrame(bank_records)

print("\n=== ORIGINAL VS TUNED PERFORMANCE BY BANK ===")

display(
    tuned_bank_comparison.sort_values(
        ["Country", "Ticker"]
    ).round(4)
)


# Measure changes in individual predictions
change_records = []

for country, group in classification_model_comparison.groupby("Country"):

    changed = int(group["Prediction_Changed"].sum())

    original_only = int((
        group["Original_Correct"]
        & ~group["Tuned_Correct"]
    ).sum())

    tuned_only = int((
        ~group["Original_Correct"]
        & group["Tuned_Correct"]
    ).sum())

    both_correct = int((
        group["Original_Correct"]
        & group["Tuned_Correct"]
    ).sum())

    both_wrong = int((
        ~group["Original_Correct"]
        & ~group["Tuned_Correct"]
    ).sum())

    assert (
        original_only + tuned_only + both_correct + both_wrong
        == len(group)
    )

    change_records.append({
        "Country": country,
        "Observations": len(group),
        "Changed_Predictions": changed,
        "Changed_Share": changed / len(group),
        "Original_Only_Correct": original_only,
        "Tuned_Only_Correct": tuned_only,
        "Both_Correct": both_correct,
        "Both_Wrong": both_wrong
    })

tuned_prediction_change_summary = pd.DataFrame(
    change_records
)

print("\n=== PREDICTION CHANGE SUMMARY ===")

display(tuned_prediction_change_summary.round(4))


# Count banks showing improvement from tuning
bank_improvement_summary = (
    tuned_bank_comparison.assign(
        Macro_F1_Improved=lambda df: df["Macro_F1_Change"] > 0,
        MCC_Improved=lambda df: df["MCC_Change"] > 0
    )
    .groupby("Country")
    .agg(
        Banks=("Ticker", "nunique"),
        Banks_Macro_F1_Improved=("Macro_F1_Improved", "sum"),
        Banks_MCC_Improved=("MCC_Improved", "sum")
    )
    .reset_index()
)

print("\n=== BANK-LEVEL IMPROVEMENT SUMMARY ===")

display(bank_improvement_summary)


# Save diagnostic results
tuned_yearly_comparison.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_yearly_comparison.csv",
    index=False
)

tuned_bank_comparison.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_bank_comparison.csv",
    index=False
)

tuned_prediction_change_summary.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_prediction_changes.csv",
    index=False
)

bank_improvement_summary.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_bank_improvement_summary.csv",
    index=False
)

print("\nCompleted.")

=== ORIGINAL VS TUNED PERFORMANCE BY YEAR ===


,Country,Year,Method,Observations,Accuracy,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Malaysia,2023,Original,2435,0.5175,0.4558,0.2238,0.3312,0.7314,0.3089
1,Malaysia,2023,Tuned,2435,0.4706,0.4445,0.2225,0.6352,0.5004,0.2559
2,Malaysia,2024,Original,2379,0.5498,0.4624,0.2554,0.2713,0.8339,0.2980
3,Malaysia,2024,Tuned,2379,0.4910,0.4466,0.2169,0.5085,0.6381,0.2401
4,Singapore,2023,Original,747,0.4485,0.4074,0.1442,0.2424,0.6698,0.3333
5,Singapore,2023,Tuned,747,0.4618,0.4175,0.1616,0.2684,0.6981,0.3081
6,Singapore,2024,Original,741,0.5250,0.4817,0.2555,0.4257,0.7059,0.3399
7,Singapore,2024,Tuned,741,0.5331,0.4807,0.2672,0.4324,0.7529,0.2964



=== ORIGINAL VS TUNED PERFORMANCE BY BANK ===


,Country,Ticker,Observations,Original_Macro_F1,Tuned_Macro_F1,Macro_F1_Change,Original_MCC,Tuned_MCC,MCC_Change
0,Malaysia,1015.KL,486,0.4383,0.4951,0.0569,0.2236,0.2881,0.0646
1,Malaysia,1023.KL,486,0.4335,0.4546,0.0211,0.1993,0.2159,0.0166
2,Malaysia,1066.KL,486,0.4902,0.3878,-0.1023,0.2881,0.1977,-0.0904
3,Malaysia,1155.KL,486,0.4687,0.3479,-0.1207,0.2409,0.1545,-0.0863
4,Malaysia,1171.KL,483,0.4613,0.4559,-0.0054,0.2726,0.2162,-0.0564
5,Malaysia,1295.KL,446,0.4776,0.4830,0.0054,0.2569,0.2709,0.0140
6,Malaysia,2488.KL,486,0.4506,0.4482,-0.0024,0.2396,0.2293,-0.0103
7,Malaysia,5185.KL,484,0.4450,0.4870,0.0420,0.2313,0.2670,0.0356
8,Malaysia,5258.KL,485,0.4380,0.4494,0.0114,0.2275,0.2036,-0.0239
9,Malaysia,5819.KL,486,0.4671,0.4076,-0.0595,0.2418,0.2136,-0.0282



=== PREDICTION CHANGE SUMMARY ===


,Country,Observations,Changed_Predictions,Changed_Share,Original_Only_Correct,Tuned_Only_Correct,Both_Correct,Both_Wrong
0,Malaysia,4814,1565,0.3251,714,460,1854,1786
1,Singapore,1488,173,0.1163,58,74,666,690



=== BANK-LEVEL IMPROVEMENT SUMMARY ===


,Country,Banks,Banks_Macro_F1_Improved,Banks_MCC_Improved
0,Malaysia,10,5,4
1,Singapore,3,2,2



Completed.


**Estimate uncertainty in original versus tuned performance**

In [21]:
BOOTSTRAP_BLOCK_WEEKS = [2, 4, 8]
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 42

bootstrap_summary_records = []
bootstrap_sample_records = []

rng = np.random.default_rng(BOOTSTRAP_SEED)

for country in ["Malaysia", "Singapore"]:

    # Align observations chronologically
    country_data = classification_model_comparison.loc[
        classification_model_comparison["Country"] == country
    ].sort_values(
        ["Date", "Ticker"]
    ).reset_index(drop=True)

    country_data["Week_Start"] = (
        country_data["Date"]
        - pd.to_timedelta(
            country_data["Date"].dt.weekday,
            unit="D"
        )
    )

    actual = country_data["Label"].to_numpy()
    original = country_data["Original_Prediction"].to_numpy()
    tuned = country_data["Tuned_Prediction"].to_numpy()

    # Calculate observed performance differences
    observed_macro_f1_difference = (
        f1_score(
            actual, tuned,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )
        - f1_score(
            actual, original,
            labels=[-1, 0, 1],
            average="macro",
            zero_division=0
        )
    )

    observed_mcc_difference = (
        matthews_corrcoef(actual, tuned)
        - matthews_corrcoef(actual, original)
    )

    # Keep all bank observations from each week together
    week_indices = list(
        country_data.groupby(
            "Week_Start",
            sort=True
        ).indices.values()
    )

    number_of_weeks = len(week_indices)

    print(f"\n=== {country.upper()} ===")
    print("Observations:", len(country_data))
    print("Prediction weeks:", number_of_weeks)

    for block_weeks in BOOTSTRAP_BLOCK_WEEKS:

        assert number_of_weeks >= block_weeks

        number_of_blocks = int(
            np.ceil(number_of_weeks / block_weeks)
        )

        macro_f1_differences = []
        mcc_differences = []

        # Resample contiguous blocks of prediction weeks
        for replicate in range(BOOTSTRAP_REPLICATES):

            block_starts = rng.integers(
                low=0,
                high=number_of_weeks - block_weeks + 1,
                size=number_of_blocks
            )

            sampled_weeks = np.concatenate([
                np.arange(start, start + block_weeks)
                for start in block_starts
            ])[:number_of_weeks]

            sampled_rows = np.concatenate([
                week_indices[week_position]
                for week_position in sampled_weeks
            ])

            sample_actual = actual[sampled_rows]
            sample_original = original[sampled_rows]
            sample_tuned = tuned[sampled_rows]

            original_f1 = f1_score(
                sample_actual,
                sample_original,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            )

            tuned_f1 = f1_score(
                sample_actual,
                sample_tuned,
                labels=[-1, 0, 1],
                average="macro",
                zero_division=0
            )

            macro_difference = tuned_f1 - original_f1

            mcc_difference = (
                matthews_corrcoef(
                    sample_actual,
                    sample_tuned
                )
                - matthews_corrcoef(
                    sample_actual,
                    sample_original
                )
            )

            macro_f1_differences.append(macro_difference)
            mcc_differences.append(mcc_difference)

            bootstrap_sample_records.append({
                "Country": country,
                "Block_Weeks": block_weeks,
                "Replicate": replicate + 1,
                "Macro_F1_Difference": macro_difference,
                "MCC_Difference": mcc_difference
            })

        macro_f1_differences = np.array(
            macro_f1_differences
        )

        mcc_differences = np.array(
            mcc_differences
        )

        # Calculate percentile intervals
        macro_bounds = np.quantile(
            macro_f1_differences,
            [0.025, 0.975]
        )

        mcc_bounds = np.quantile(
            mcc_differences,
            [0.025, 0.975]
        )

        bootstrap_summary_records.append({
            "Country": country,
            "Block_Weeks": block_weeks,
            "Replicates": BOOTSTRAP_REPLICATES,
            "Observations": len(country_data),
            "Prediction_Weeks": number_of_weeks,
            "Observed_Macro_F1_Difference": (
                observed_macro_f1_difference
            ),
            "Macro_F1_CI_Lower": macro_bounds[0],
            "Macro_F1_CI_Upper": macro_bounds[1],
            "Bootstrap_Share_Macro_F1_Positive": (
                np.mean(macro_f1_differences > 0)
            ),
            "Observed_MCC_Difference": (
                observed_mcc_difference
            ),
            "MCC_CI_Lower": mcc_bounds[0],
            "MCC_CI_Upper": mcc_bounds[1]
        })

        print(
            f"Block length: {block_weeks} weeks | "
            f"Macro F1 interval: "
            f"[{macro_bounds[0]:.4f}, "
            f"{macro_bounds[1]:.4f}]"
        )

# Assemble bootstrap outputs
classification_bootstrap_summary = pd.DataFrame(
    bootstrap_summary_records
)

classification_bootstrap_samples = pd.DataFrame(
    bootstrap_sample_records
)

print("\n=== PAIRED BLOCK-BOOTSTRAP RESULTS ===")

display(
    classification_bootstrap_summary.round(4)
)

print("\n=== INTERVAL INTERPRETATION ===")

interpretation = (
    classification_bootstrap_summary[
        [
            "Country",
            "Block_Weeks",
            "Macro_F1_CI_Lower",
            "Macro_F1_CI_Upper",
            "MCC_CI_Lower",
            "MCC_CI_Upper"
        ]
    ].copy()
)

interpretation["Macro_F1_Excludes_Zero"] = (
    (interpretation["Macro_F1_CI_Lower"] > 0)
    | (interpretation["Macro_F1_CI_Upper"] < 0)
)

interpretation["MCC_Excludes_Zero"] = (
    (interpretation["MCC_CI_Lower"] > 0)
    | (interpretation["MCC_CI_Upper"] < 0)
)

display(interpretation.round(4))

# Save uncertainty estimates
classification_bootstrap_summary.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_block_bootstrap_summary.csv",
    index=False
)

classification_bootstrap_samples.to_csv(
    RESULTS_DIR / "triple_barrier_tuning_block_bootstrap_samples.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Observations: 4814
Prediction weeks: 104
Block length: 2 weeks | Macro F1 interval: [-0.0353, 0.0115]
Block length: 4 weeks | Macro F1 interval: [-0.0362, 0.0101]
Block length: 8 weeks | Macro F1 interval: [-0.0335, 0.0075]

=== SINGAPORE ===
Observations: 1488
Prediction weeks: 104
Block length: 2 weeks | Macro F1 interval: [-0.0148, 0.0250]
Block length: 4 weeks | Macro F1 interval: [-0.0134, 0.0234]
Block length: 8 weeks | Macro F1 interval: [-0.0128, 0.0227]

=== PAIRED BLOCK-BOOTSTRAP RESULTS ===


,Country,Block_Weeks,Replicates,Observations,Prediction_Weeks,Observed_Macro_F1_Difference,Macro_F1_CI_Lower,Macro_F1_CI_Upper,Bootstrap_Share_Macro_F1_Positive,Observed_MCC_Difference,MCC_CI_Lower,MCC_CI_Upper
0,Malaysia,2,1000,4814,104,-0.0126,-0.0353,0.0115,0.133,-0.0188,-0.0467,0.0097
1,Malaysia,4,1000,4814,104,-0.0126,-0.0362,0.0101,0.137,-0.0188,-0.0500,0.0146
2,Malaysia,8,1000,4814,104,-0.0126,-0.0335,0.0075,0.128,-0.0188,-0.0501,0.0147
3,Singapore,2,1000,1488,104,0.0046,-0.0148,0.0250,0.659,0.0139,-0.0116,0.0412
4,Singapore,4,1000,1488,104,0.0046,-0.0134,0.0234,0.682,0.0139,-0.0105,0.0415
5,Singapore,8,1000,1488,104,0.0046,-0.0128,0.0227,0.701,0.0139,-0.0092,0.0375



=== INTERVAL INTERPRETATION ===


,Country,Block_Weeks,Macro_F1_CI_Lower,Macro_F1_CI_Upper,MCC_CI_Lower,MCC_CI_Upper,Macro_F1_Excludes_Zero,MCC_Excludes_Zero
0,Malaysia,2,-0.0353,0.0115,-0.0467,0.0097,False,False
1,Malaysia,4,-0.0362,0.0101,-0.0500,0.0146,False,False
2,Malaysia,8,-0.0335,0.0075,-0.0501,0.0147,False,False
3,Singapore,2,-0.0148,0.0250,-0.0116,0.0412,False,False
4,Singapore,4,-0.0134,0.0234,-0.0105,0.0415,False,False
5,Singapore,8,-0.0128,0.0227,-0.0092,0.0375,False,False



Completed.


**Estimate accuracy differences with paired block bootstrapping**

In [22]:
ACCURACY_BOOTSTRAP_BLOCKS = [2, 4, 8]
ACCURACY_BOOTSTRAP_REPLICATES = 1000

rng = np.random.default_rng(42)

accuracy_bootstrap_records = []

for country in ["Malaysia", "Singapore"]:

    # Arrange observations by prediction date
    country_data = classification_model_comparison.loc[
        classification_model_comparison["Country"] == country
    ].sort_values(
        ["Date", "Ticker"]
    ).reset_index(drop=True)

    country_data["Week_Start"] = (
        country_data["Date"]
        - pd.to_timedelta(
            country_data["Date"].dt.weekday,
            unit="D"
        )
    )

    actual = country_data["Label"].to_numpy()
    original = country_data["Original_Prediction"].to_numpy()
    tuned = country_data["Tuned_Prediction"].to_numpy()

    # Select the provisional model for each market
    selected = original if country == "Malaysia" else tuned

    # Construct the historical majority-class baseline
    flat_baseline = np.zeros(len(actual), dtype=int)

    # Keep observations from the same week together
    weekly_indices = list(
        country_data.groupby(
            "Week_Start",
            sort=True
        ).indices.values()
    )

    number_of_weeks = len(weekly_indices)

    comparisons = {
        "Tuned_minus_Original": (tuned, original),
        "Selected_minus_Flat": (selected, flat_baseline)
    }

    print(f"\n=== {country.upper()} ===")
    print("Prediction weeks:", number_of_weeks)

    for block_weeks in ACCURACY_BOOTSTRAP_BLOCKS:

        number_of_blocks = int(
            np.ceil(number_of_weeks / block_weeks)
        )

        bootstrap_differences = {
            name: [] for name in comparisons
        }

        # Resample consecutive blocks of prediction weeks
        for replicate in range(ACCURACY_BOOTSTRAP_REPLICATES):

            starts = rng.integers(
                0,
                number_of_weeks - block_weeks + 1,
                size=number_of_blocks
            )

            sampled_weeks = np.concatenate([
                np.arange(start, start + block_weeks)
                for start in starts
            ])[:number_of_weeks]

            sampled_rows = np.concatenate([
                weekly_indices[week_position]
                for week_position in sampled_weeks
            ])

            sampled_actual = actual[sampled_rows]

            for name, (first, second) in comparisons.items():

                first_accuracy = np.mean(
                    first[sampled_rows] == sampled_actual
                )

                second_accuracy = np.mean(
                    second[sampled_rows] == sampled_actual
                )

                bootstrap_differences[name].append(
                    100 * (first_accuracy - second_accuracy)
                )

        # Calculate observed differences and uncertainty intervals
        for name, (first, second) in comparisons.items():

            observed_difference = 100 * (
                np.mean(first == actual)
                - np.mean(second == actual)
            )

            differences = np.array(
                bootstrap_differences[name]
            )

            lower, upper = np.quantile(
                differences,
                [0.025, 0.975]
            )

            accuracy_bootstrap_records.append({
                "Country": country,
                "Comparison": name,
                "Block_Weeks": block_weeks,
                "Replicates": ACCURACY_BOOTSTRAP_REPLICATES,
                "Observed_Difference_pp": observed_difference,
                "CI_Lower_pp": lower,
                "CI_Upper_pp": upper,
                "Interval_Excludes_Zero": (
                    lower > 0 or upper < 0
                )
            })

# Combine and display results
accuracy_bootstrap_results = pd.DataFrame(
    accuracy_bootstrap_records
)

print("\n=== ACCURACY DIFFERENCE BOOTSTRAP RESULTS ===")

display(
    accuracy_bootstrap_results.sort_values(
        ["Country", "Comparison", "Block_Weeks"]
    ).round(4)
)

# Display main four-week analysis separately
print("\n=== MAIN FOUR-WEEK BLOCK RESULTS ===")

display(
    accuracy_bootstrap_results.loc[
        accuracy_bootstrap_results["Block_Weeks"] == 4
    ].round(4)
)

# Save the uncertainty analysis
accuracy_bootstrap_results.to_csv(
    RESULTS_DIR / "triple_barrier_accuracy_bootstrap_results.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Prediction weeks: 104

=== SINGAPORE ===
Prediction weeks: 104

=== ACCURACY DIFFERENCE BOOTSTRAP RESULTS ===


,Country,Comparison,Block_Weeks,Replicates,Observed_Difference_pp,CI_Lower_pp,CI_Upper_pp,Interval_Excludes_Zero
1,Malaysia,Selected_minus_Flat,2,1000,5.2971,3.2709,7.3888,True
3,Malaysia,Selected_minus_Flat,4,1000,5.2971,2.9712,7.3428,True
5,Malaysia,Selected_minus_Flat,8,1000,5.2971,3.3656,7.3237,True
0,Malaysia,Tuned_minus_Original,2,1000,-5.2763,-7.6130,-2.8226,True
2,Malaysia,Tuned_minus_Original,4,1000,-5.2763,-7.6972,-2.9620,True
4,Malaysia,Tuned_minus_Original,8,1000,-5.2763,-7.3868,-3.1868,True
7,Singapore,Selected_minus_Flat,2,1000,5.5108,0.8146,9.5054,True
9,Singapore,Selected_minus_Flat,4,1000,5.5108,0.6677,9.9138,True
11,Singapore,Selected_minus_Flat,8,1000,5.5108,1.2023,9.8290,True
6,Singapore,Tuned_minus_Original,2,1000,1.0753,-0.5990,2.8611,False



=== MAIN FOUR-WEEK BLOCK RESULTS ===


,Country,Comparison,Block_Weeks,Replicates,Observed_Difference_pp,CI_Lower_pp,CI_Upper_pp,Interval_Excludes_Zero
2,Malaysia,Tuned_minus_Original,4,1000,-5.2763,-7.6972,-2.9620,True
3,Malaysia,Selected_minus_Flat,4,1000,5.2971,2.9712,7.3428,True
8,Singapore,Tuned_minus_Original,4,1000,1.0753,-0.4049,2.6625,False
9,Singapore,Selected_minus_Flat,4,1000,5.5108,0.6677,9.9138,True



Completed.


**Audit MBSB prices around the June 2016 rights issue**

In [23]:
MBSB_TICKER = "1171.KL"
RIGHTS_EX_DATE = pd.Timestamp("2016-06-21")

# Load the cleaned historical price data
audit_prices = pd.read_csv(
    PROCESSED_DIR / "lseg_ohlcv_clean.csv",
    parse_dates=["Date"]
)

# Select MBSB observations in chronological order
mbsb_prices = audit_prices.loc[
    audit_prices["Ticker"] == MBSB_TICKER
].sort_values("Date").reset_index(drop=True)

event_positions = np.flatnonzero(
    mbsb_prices["Date"].eq(RIGHTS_EX_DATE).to_numpy()
)

assert len(event_positions) == 1

event_position = int(event_positions[0])

assert event_position > 0

previous_row = mbsb_prices.iloc[event_position - 1]
event_row = mbsb_prices.iloc[event_position]

assert (
    previous_row["Segment_ID"] == event_row["Segment_ID"]
)

# Verify the observed daily price change
calculated_return = (
    event_row["Close"] / previous_row["Close"] - 1
)

assert np.isclose(
    calculated_return,
    event_row["returns"],
    atol=1e-8
)

print("=== MBSB RIGHTS ISSUE DATE ===")
print("Ex-date:", RIGHTS_EX_DATE.date())
print("Previous trading date:", previous_row["Date"].date())
print("Previous close:", round(previous_row["Close"], 4))
print("Ex-date close:", round(event_row["Close"], 4))
print("Observed return (%):", round(calculated_return * 100, 4))

# Display ten sessions before and after the ex-date
event_window = mbsb_prices.iloc[
    max(0, event_position - 10):
    min(len(mbsb_prices), event_position + 11)
].copy()

event_window["Return_pct"] = (
    event_window["returns"] * 100
)

print("\n=== MBSB PRICE WINDOW ===")

display(
    event_window[
        [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Return_pct"
        ]
    ].round(4)
)

# Compare Malaysian bank returns on the ex-date
market_day = audit_prices.loc[
    (audit_prices["Country"] == "Malaysia")
    & (audit_prices["Date"] == RIGHTS_EX_DATE)
].copy()

market_day["Return_pct"] = market_day["returns"] * 100

print("\n=== MALAYSIAN BANK RETURNS ON 21 JUNE 2016 ===")

display(
    market_day[
        ["Ticker", "Close", "Return_pct"]
    ].sort_values("Return_pct").round(4)
)

peer_returns = market_day.loc[
    market_day["Ticker"] != MBSB_TICKER,
    "Return_pct"
].dropna()

print("Other banks' median return (%):",
      round(peer_returns.median(), 4))

# Inspect the largest MBSB historical daily movements
historical_mbsb = mbsb_prices.loc[
    mbsb_prices["Date"] <= "2022-12-31"
].copy()

historical_mbsb["Absolute_Return"] = (
    historical_mbsb["returns"].abs()
)

historical_mbsb["Return_pct"] = (
    historical_mbsb["returns"] * 100
)

print("\n=== LARGEST MBSB DAILY RETURNS IN TRAINING HISTORY ===")

display(
    historical_mbsb.sort_values(
        "Absolute_Return",
        ascending=False
    )[
        ["Date", "Close", "Return_pct", "Volume"]
    ].head(12).round(4)
)

# Identify classification observations near the rights issue
nearby_dates = mbsb_prices.iloc[
    max(0, event_position - 5):
    min(len(mbsb_prices), event_position + 21)
]["Date"]

nearby_labels = classification_tuning_data.loc[
    (classification_tuning_data["Ticker"] == MBSB_TICKER)
    & (classification_tuning_data["Date"].isin(nearby_dates))
    & (classification_tuning_data["Date"] <= "2022-12-31")
].copy()

print("\n=== CLASSIFICATION OBSERVATIONS NEAR THE EX-DATE ===")

print("Nearby labelled observations:", len(nearby_labels))

display(
    nearby_labels[
        ["Date", "Event_End_Date", "Label"]
    ].sort_values("Date")
)

print("\n=== NEARBY LABEL DISTRIBUTION ===")

display(
    nearby_labels["Label"].value_counts()
    .reindex([-1, 0, 1], fill_value=0)
    .rename_axis("Label")
    .reset_index(name="Observations")
)

# Save the corporate-action audit
event_window.to_csv(
    RESULTS_DIR / "mbsb_2016_rights_issue_price_window.csv",
    index=False
)

market_day.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_market_comparison.csv",
    index=False
)

nearby_labels.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_nearby_labels.csv",
    index=False
)

print("\nCompleted.")

=== MBSB RIGHTS ISSUE DATE ===
Ex-date: 2016-06-21
Previous trading date: 2016-06-20
Previous close: 1.035
Ex-date close: 0.795
Observed return (%): -23.1884

=== MBSB PRICE WINDOW ===


C:\Users\zechi\AppData\Local\Temp\ipykernel_15704\3549306222.py:73: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(4)


,Date,Open,High,Low,Close,Volume,Return_pct
352,2016-06-07,1.1124,1.1607,1.1124,1.1607,9.405659e+05,3.4483
353,2016-06-08,1.1704,1.1704,1.1511,1.1607,5.176317e+05,0.0000
354,2016-06-09,1.1511,1.1607,1.1414,1.1414,2.854366e+05,-1.6667
355,2016-06-10,1.1414,1.1607,1.1221,1.1221,1.919176e+06,-1.6949
356,2016-06-13,1.1221,1.1221,1.1027,1.1027,1.327937e+06,-1.7241
357,2016-06-14,1.1124,1.1124,1.0930,1.1027,1.223004e+06,0.0000
358,2016-06-15,1.1027,1.1027,1.0834,1.0930,2.483950e+06,-0.8772
359,2016-06-16,1.0930,1.0930,1.0640,1.0640,2.309028e+06,-2.6549
360,2016-06-17,1.0640,1.0834,1.0447,1.0543,3.396603e+06,-0.9091
361,2016-06-20,1.0543,1.0543,1.0157,1.0350,6.272886e+06,-1.8349



=== MALAYSIAN BANK RETURNS ON 21 JUNE 2016 ===


,Ticker,Close,Return_pct
11921,1171.KL,0.7950,-23.1884
3254,1023.KL,4.1334,-0.4630
26360,5819.KL,13.2600,0.1511
9025,1155.KL,8.2200,0.2439
14816,1295.KL,3.8480,0.4175
362,1015.KL,4.4400,0.9091
20597,5185.KL,1.8751,0.9390
17710,2488.KL,4.0048,1.4815
23482,5258.KL,3.0839,1.7284


Other banks' median return (%): 0.6633

=== LARGEST MBSB DAILY RETURNS IN TRAINING HISTORY ===


... (3 outputs left out)

,Date,Event_End_Date,Label
3228,2016-06-14,2016-06-16,-1
3237,2016-06-15,2016-06-16,-1
3246,2016-06-16,2016-06-20,-1
3255,2016-06-17,2016-06-21,-1
3263,2016-06-20,2016-06-21,-1
3271,2016-06-21,2016-06-29,0
3279,2016-06-23,2016-06-30,0
3287,2016-06-24,2016-07-01,0
3295,2016-06-27,2016-07-04,0
3303,2016-06-28,2016-07-05,0



=== NEARBY LABEL DISTRIBUTION ===


,Label,Observations
0,-1,10
1,0,14
2,1,2



Completed.


**Test MBSB rights-issue sensitivity in fixed classification training**

In [24]:
MBSB_PRE_EVENT_SESSIONS = 5
MBSB_POST_EVENT_SESSIONS = 60

mbsb_event_index = int(
    np.flatnonzero(
        mbsb_prices["Date"].eq(RIGHTS_EX_DATE).to_numpy()
    )[0]
)

exclusion_start = mbsb_prices.iloc[
    mbsb_event_index - MBSB_PRE_EVENT_SESSIONS
]["Date"]

exclusion_end = mbsb_prices.iloc[
    mbsb_event_index + MBSB_POST_EVENT_SESSIONS
]["Date"]

# Prepare the original fixed training dataset
malaysia_fixed_train = final_classification_train.loc[
    final_classification_train["Country"] == "Malaysia"
].copy()

malaysia_fixed_valid = final_classification_valid.loc[
    final_classification_valid["Country"] == "Malaysia"
].copy()

# Identify training observations potentially influenced by the event
event_exclusion_mask = (
    malaysia_fixed_train["Ticker"].eq(MBSB_TICKER)
    & malaysia_fixed_train["Date"].between(
        exclusion_start,
        exclusion_end
    )
)

malaysia_filtered_train = malaysia_fixed_train.loc[
    ~event_exclusion_mask
].copy()

print("=== EXCLUSION WINDOW ===")
print("Start:", exclusion_start.date())
print("Ex-date:", RIGHTS_EX_DATE.date())
print("End:", exclusion_end.date())
print("Excluded observations:", int(event_exclusion_mask.sum()))
print("Original training rows:", len(malaysia_fixed_train))
print("Filtered training rows:", len(malaysia_filtered_train))

assert event_exclusion_mask.sum() > 0
assert len(malaysia_filtered_train) > 0

assert (
    malaysia_filtered_train["Event_End_Date"].max()
    <= FINAL_TRAIN_END
)

# Fit identical models with and without the event window
sensitivity_predictions = {}

training_variants = {
    "Full_Training": malaysia_fixed_train,
    "Exclude_Rights_Window": malaysia_filtered_train
}

for variant_name, training_subset in training_variants.items():

    model = XGBClassifier(**model_params)

    model.fit(
        training_subset[feature_columns],
        training_subset["Label"].to_numpy() + 1,
        verbose=False
    )

    predictions = (
        model.predict(
            malaysia_fixed_valid[feature_columns]
        ) - 1
    )

    sensitivity_predictions[variant_name] = predictions

    print(
        variant_name,
        "| Training rows:",
        len(training_subset)
    )

# Align the historical fixed and rolling model references
reference_predictions = corrected_predictions.loc[
    (corrected_predictions["Country"] == "Malaysia")
    & corrected_predictions["Strategy"].isin(
        ["Fixed", "Rolling_156W"]
    ),
    [
        "Ticker",
        "Date",
        "Label",
        "Strategy",
        "Predicted_Label"
    ]
].copy()

reference_wide = reference_predictions.pivot(
    index=["Ticker", "Date", "Label"],
    columns="Strategy",
    values="Predicted_Label"
).reset_index()

comparison = malaysia_fixed_valid[
    ["Ticker", "Date", "Label"]
].copy()

comparison["Full_Training"] = (
    sensitivity_predictions["Full_Training"]
)

comparison["Exclude_Rights_Window"] = (
    sensitivity_predictions["Exclude_Rights_Window"]
)

comparison = comparison.merge(
    reference_wide,
    on=["Ticker", "Date", "Label"],
    how="left",
    validate="one_to_one"
)

assert len(comparison) == len(malaysia_fixed_valid)

assert comparison[
    [
        "Fixed",
        "Rolling_156W",
        "Full_Training",
        "Exclude_Rights_Window"
    ]
].notna().all().all()

# Evaluate fixed-model sensitivity and the rolling reference
sensitivity_result_records = []

model_columns = {
    "Original_Fixed_Reference": "Fixed",
    "Full_Training_Refit": "Full_Training",
    "Exclude_Rights_Window": "Exclude_Rights_Window",
    "Original_Rolling_156W": "Rolling_156W"
}

for model_name, prediction_column in model_columns.items():

    y_true = comparison["Label"].to_numpy()
    y_pred = comparison[prediction_column].to_numpy()

    metrics = calculate_classification_metrics(
        comparison.rename(
            columns={prediction_column: "Evaluation_Prediction"}
        ),
        "Evaluation_Prediction"
    )

    sensitivity_result_records.append({
        "Model": model_name,
        "Observations": len(comparison),
        **metrics,
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        )
    })

mbsb_fixed_sensitivity_results = pd.DataFrame(
    sensitivity_result_records
)

print("\n=== MALAYSIAN FIXED-MODEL SENSITIVITY ===")

display(
    mbsb_fixed_sensitivity_results[
        [
            "Model",
            "Observations",
            "Accuracy",
            "Balanced_Accuracy",
            "Macro_F1",
            "MCC",
            "Down_Recall",
            "Flat_Recall",
            "Up_Recall"
        ]
    ].round(4)
)

# Count prediction changes caused by excluding the event window
refit_changed = (
    comparison["Full_Training"]
    != comparison["Exclude_Rights_Window"]
)

reference_agreement = (
    comparison["Fixed"]
    == comparison["Full_Training"]
)

print("\n=== PREDICTION STABILITY ===")

stability_summary = pd.DataFrame([
    {
        "Comparison": "Full_Refit_vs_Original_Reference",
        "Changed_Predictions": int(
            (~reference_agreement).sum()
        ),
        "Changed_Share": float(
            (~reference_agreement).mean()
        )
    },
    {
        "Comparison": "Event_Excluded_vs_Full_Refit",
        "Changed_Predictions": int(
            refit_changed.sum()
        ),
        "Changed_Share": float(
            refit_changed.mean()
        )
    }
])

display(stability_summary.round(4))

# Check the affected bank separately
print("\n=== MBSB VALIDATION PERFORMANCE ===")

mbsb_validation = comparison.loc[
    comparison["Ticker"] == MBSB_TICKER
].copy()

mbsb_result_records = []

for model_name, prediction_column in model_columns.items():

    metrics = calculate_classification_metrics(
        mbsb_validation,
        prediction_column
    )

    mbsb_result_records.append({
        "Model": model_name,
        "Observations": len(mbsb_validation),
        "Accuracy": metrics["Accuracy"],
        "Macro_F1": metrics["Macro_F1"],
        "MCC": metrics["MCC"]
    })

mbsb_validation_results = pd.DataFrame(
    mbsb_result_records
)

display(mbsb_validation_results.round(4))

# Save sensitivity results
mbsb_fixed_sensitivity_results.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_fixed_model_sensitivity.csv",
    index=False
)

stability_summary.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_prediction_stability.csv",
    index=False
)

mbsb_validation_results.to_csv(
    RESULTS_DIR / "mbsb_rights_issue_bank_validation_sensitivity.csv",
    index=False
)

print("\nCompleted.")

=== EXCLUSION WINDOW ===
Start: 2016-06-14
Ex-date: 2016-06-21
End: 2016-09-21
Excluded observations: 66
Original training rows: 18519
Filtered training rows: 18453
Full_Training | Training rows: 18519
Exclude_Rights_Window | Training rows: 18453

=== MALAYSIAN FIXED-MODEL SENSITIVITY ===


,Model,Observations,Accuracy,Balanced_Accuracy,Macro_F1,MCC,Down_Recall,Flat_Recall,Up_Recall
0,Original_Fixed_Reference,4814,0.5407,0.4675,0.4407,0.2556,0.4436,0.8284,0.1305
1,Full_Training_Refit,4814,0.5407,0.4681,0.4417,0.2556,0.4453,0.8262,0.1327
2,Exclude_Rights_Window,4814,0.5442,0.4733,0.4499,0.2616,0.4488,0.8227,0.1483
3,Original_Rolling_156W,4814,0.5334,0.4630,0.4600,0.2387,0.3038,0.7821,0.3032



=== PREDICTION STABILITY ===


,Comparison,Changed_Predictions,Changed_Share
0,Full_Refit_vs_Original_Reference,221,0.0459
1,Event_Excluded_vs_Full_Refit,223,0.0463



=== MBSB VALIDATION PERFORMANCE ===


,Model,Observations,Accuracy,Macro_F1,MCC
0,Original_Fixed_Reference,483,0.5611,0.4457,0.2641
1,Full_Training_Refit,483,0.5631,0.4513,0.2689
2,Exclude_Rights_Window,483,0.5590,0.4483,0.2620
3,Original_Rolling_156W,483,0.5673,0.4613,0.2726



Completed.


**Verify selected model prediction reproducibility**

In [25]:
reproduction_records = []
reproduction_details = []

malaysia_weeks = sorted(
    final_classification_valid.loc[
        final_classification_valid["Country"] == "Malaysia",
        "Week_Start"
    ].unique()
)

pilot_positions = sorted(set([
    0,
    len(malaysia_weeks) // 4,
    len(malaysia_weeks) // 2,
    3 * len(malaysia_weeks) // 4,
    len(malaysia_weeks) - 1
]))

malaysia_pilot_weeks = [
    malaysia_weeks[position]
    for position in pilot_positions
]

for country in ["Malaysia", "Singapore"]:

    country_valid = final_classification_valid.loc[
        final_classification_valid["Country"] == country
    ].copy()

    if country == "Malaysia":

        strategy = "Rolling_156W"
        candidate_name = "Original"

        prediction_groups = [
            (
                week_start,
                country_valid.loc[
                    country_valid["Week_Start"] == week_start
                ].copy()
            )
            for week_start in malaysia_pilot_weeks
        ]

        reference_predictions = corrected_predictions.loc[
            (corrected_predictions["Country"] == country)
            & (corrected_predictions["Strategy"] == strategy),
            ["Ticker", "Date", "Label", "Predicted_Label"]
        ].copy()

    else:

        strategy = "Fixed"
        candidate_name = "Depth3_Regularised"

        prediction_groups = [
            (EXTERNAL_VALIDATION_START, country_valid)
        ]

        reference_predictions = (
            tuned_classification_predictions.loc[
                tuned_classification_predictions["Country"] == country,
                ["Ticker", "Date", "Label", "Predicted_Label"]
            ].copy()
        )

    candidate = next(
        item for item in classification_candidates
        if item["Name"] == candidate_name
    )

    parameters = {
        **model_params,
        **candidate["Overrides"]
    }

    print(f"\n=== {country.upper()} ===")
    print("Candidate:", candidate_name)
    print("Training strategy:", strategy)

    for week_start, week_data in prediction_groups:

        if country == "Malaysia":

            rolling_start = (
                week_start - pd.Timedelta(weeks=156)
            )

            train = classification_tuning_data.loc[
                (classification_tuning_data["Country"] == country)
                & (classification_tuning_data["Date"] >= rolling_start)
                & (classification_tuning_data["Date"] < week_start)
                & (
                    classification_tuning_data["Event_End_Date"]
                    < week_start
                )
            ].copy()

        else:

            train = final_classification_train.loc[
                final_classification_train["Country"] == country
            ].copy()

        assert len(train) > 0
        assert train["Event_End_Date"].max() < week_data["Date"].min()

        # Refit using the documented training procedure
        model = XGBClassifier(**parameters)

        model.fit(
            train[feature_columns],
            train["Label"].to_numpy() + 1,
            verbose=False
        )

        reproduced = (
            model.predict(week_data[feature_columns]) - 1
        )

        # Match saved predictions to the same stock-date observations
        comparison = week_data[
            ["Ticker", "Date", "Label"]
        ].copy()

        comparison["Reproduced_Prediction"] = reproduced

        reference = reference_predictions.rename(
            columns={
                "Predicted_Label": "Recorded_Prediction"
            }
        )

        comparison = comparison.merge(
            reference,
            on=["Ticker", "Date", "Label"],
            how="left",
            validate="one_to_one"
        )

        assert comparison["Recorded_Prediction"].notna().all()

        comparison["Country"] = country
        comparison["Strategy"] = strategy
        comparison["Candidate"] = candidate_name
        comparison["Training_Week"] = week_start

        comparison["Predictions_Match"] = (
            comparison["Recorded_Prediction"]
            == comparison["Reproduced_Prediction"]
        )

        reproduction_details.append(comparison)

        reproduction_records.append({
            "Country": country,
            "Strategy": strategy,
            "Candidate": candidate_name,
            "Training_Week": week_start,
            "Training_Rows": len(train),
            "Prediction_Rows": len(comparison),
            "Matching_Predictions": int(
                comparison["Predictions_Match"].sum()
            ),
            "Different_Predictions": int(
                (~comparison["Predictions_Match"]).sum()
            ),
            "Agreement": comparison["Predictions_Match"].mean(),
            "Recorded_Accuracy": accuracy_score(
                comparison["Label"],
                comparison["Recorded_Prediction"]
            ),
            "Reproduced_Accuracy": accuracy_score(
                comparison["Label"],
                comparison["Reproduced_Prediction"]
            )
        })

        print(
            f"Week {pd.Timestamp(week_start).date()} | "
            f"Training: {len(train)} | "
            f"Predictions: {len(comparison)} | "
            f"Agreement: "
            f"{comparison['Predictions_Match'].mean():.4f}"
        )

# Summarise reproducibility
classification_reproduction_results = pd.DataFrame(
    reproduction_records
)

classification_reproduction_details = pd.concat(
    reproduction_details,
    ignore_index=True
)

print("\n=== REPRODUCTION BY TRAINING GROUP ===")

display(
    classification_reproduction_results.round(4)
)

print("\n=== REPRODUCTION SUMMARY ===")

reproduction_summary = (
    classification_reproduction_results
    .groupby(["Country", "Candidate", "Strategy"])
    .agg(
        Training_Groups=("Training_Week", "size"),
        Prediction_Rows=("Prediction_Rows", "sum"),
        Matching_Predictions=("Matching_Predictions", "sum"),
        Different_Predictions=("Different_Predictions", "sum")
    )
    .reset_index()
)

reproduction_summary["Agreement"] = (
    reproduction_summary["Matching_Predictions"]
    / reproduction_summary["Prediction_Rows"]
)

display(reproduction_summary.round(4))

# Save audit results
classification_reproduction_results.to_csv(
    RESULTS_DIR / "triple_barrier_model_reproduction_groups.csv",
    index=False
)

classification_reproduction_details.to_csv(
    RESULTS_DIR / "triple_barrier_model_reproduction_details.csv",
    index=False
)

reproduction_summary.to_csv(
    RESULTS_DIR / "triple_barrier_model_reproduction_summary.csv",
    index=False
)

print("\nCompleted.")


=== MALAYSIA ===
Candidate: Original
Training strategy: Rolling_156W
Week 2023-01-02 | Training: 7174 | Predictions: 40 | Agreement: 1.0000
Week 2023-07-03 | Training: 7126 | Predictions: 50 | Agreement: 1.0000
Week 2024-01-01 | Training: 7142 | Predictions: 39 | Agreement: 1.0000
Week 2024-07-01 | Training: 7191 | Predictions: 50 | Agreement: 1.0000
Week 2024-12-23 | Training: 7253 | Predictions: 10 | Agreement: 1.0000

=== SINGAPORE ===
Candidate: Depth3_Regularised
Training strategy: Fixed
Week 2023-01-01 | Training: 5908 | Predictions: 1488 | Agreement: 1.0000

=== REPRODUCTION BY TRAINING GROUP ===


C:\Users\zechi\AppData\Local\Temp\ipykernel_15704\45376119.py:200: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  classification_reproduction_results.round(4)


,Country,Strategy,Candidate,Training_Week,Training_Rows,Prediction_Rows,Matching_Predictions,Different_Predictions,Agreement,Recorded_Accuracy,Reproduced_Accuracy
0,Malaysia,Rolling_156W,Original,2023-01-02,7174,40,40,0,1.0,0.7000,0.7000
1,Malaysia,Rolling_156W,Original,2023-07-03,7126,50,50,0,1.0,0.6400,0.6400
2,Malaysia,Rolling_156W,Original,2024-01-01,7142,39,39,0,1.0,0.4872,0.4872
3,Malaysia,Rolling_156W,Original,2024-07-01,7191,50,50,0,1.0,0.4400,0.4400
4,Malaysia,Rolling_156W,Original,2024-12-23,7253,10,10,0,1.0,0.7000,0.7000
5,Singapore,Fixed,Depth3_Regularised,2023-01-01,5908,1488,1488,0,1.0,0.4973,0.4973



=== REPRODUCTION SUMMARY ===


,Country,Candidate,Strategy,Training_Groups,Prediction_Rows,Matching_Predictions,Different_Predictions,Agreement
0,Malaysia,Original,Rolling_156W,5,189,189,0,1.0
1,Singapore,Depth3_Regularised,Fixed,1,1488,1488,0,1.0



Completed.
